# CALM-Sep Stage 2 — Universal Adapter Evaluation

Measures SI-SDRi of the universal adapter vs. bare SR-CorrNet baseline on
four conditions: clean, reverb, noise, codec.

Input datasets needed:
- `calmsep-8k-slice` (audio + rirs + noise)
- `calmsep-model` (sr_corrnet source + HF cache)
- Stage 2 checkpoint: either set `STAGE2_CKPT` below or ensure
  the training notebook output is accessible at the path shown.

Expected runtime: ~15 min on T4 (50 samples × 4 conditions).


In [ ]:
AUDIO      = '/kaggle/input/datasets/rishig777/calmsep-8k-slice/calmsep-kaggle'
MODEL_DS   = '/kaggle/input/datasets/rishig777/calmsep-model/calmsep-tiny'
SRCORRNET  = f'{MODEL_DS}/sr_corrnet_src'
HF_CACHE   = f'{MODEL_DS}/hf_cache'
PROJ       = '/tmp/calmsep_project'
WORK       = '/kaggle/working'
# Path to Stage 2 best_universal.pt  (update if saved as a separate dataset)
STAGE2_CKPT = '/kaggle/input/datasets/rishig777/calmsep-stage2-ckpt/best_universal.pt'
N_EVAL_PER_COND = 50   # samples per condition (50 × 4 = 200 total)
DEVICE = 'cuda'
import os
os.environ['HF_HOME']              = HF_CACHE
os.environ['HF_HUB_OFFLINE']       = '1'
os.environ['TRANSFORMERS_OFFLINE']  = '1'
os.environ['HF_DATASETS_OFFLINE']   = '1'
print('audio:',    os.path.exists(AUDIO))
print('hf_cache:', os.path.exists(HF_CACHE))
print('stage2_ckpt exists:', os.path.exists(STAGE2_CKPT))


In [ ]:
import os, base64
PROJ = '/tmp/calmsep_project'
os.makedirs(f'{PROJ}/train', exist_ok=True)
open(f'{PROJ}/train/__init__.py', 'wb').write(base64.b64decode('IiIiVHJhaW5pbmcgbG9vcHMgYW5kIGNvbXBvc2l0ZSBsb3NzIGFzc2VtYmx5IChEZXYgQikuIiIiCg=='))
os.makedirs(f'{PROJ}/train', exist_ok=True)
open(f'{PROJ}/train/stage1_single.py', 'wb').write(base64.b64decode('...'))
os.makedirs(f'{PROJ}/train', exist_ok=True)
open(f'{PROJ}/train/losses.py', 'wb').write(base64.b64decode('IiIiClNoYXJlZCBsb3NzIGZ1bmN0aW9ucyBmb3IgQ0FMTS1TZXAgYWRhcHRlciB0cmFpbmluZyAoRGV2IEIsIFAxLUIzKS4KClJldXNlcyB0aGUgYmFja2JvbmUgZW5naW5lIGxvc3NlcyB3aGVyZSBwb3NzaWJsZToKICBwcmltYXJ5OiBQSVQgU0ktU05SIG9uIHdhdmVmb3JtcyAodGltZSBkb21haW4pCiAgc2Vjb25kYXJ5OiAwLjUgw5cgUElUIFNJLVNOUiBvbiBtYWduaXR1ZGUgU1RGVCAoZnJlcXVlbmN5IGRvbWFpbikKICBhdHRyYWN0b3I6IEJDRSBvbiBwcmVzWyJsb2dpdHMiXSAoc2hhcGUgMSw3KQoKQ2FyZGluYWxpdHktYXdhcmUgZXh0ZW5zaW9uIChCTFVFUFJJTlQgwqc4LjIpOgogIE1pc3NlZCBzcGVha2VycyBzY29yZSAwIGRCLiBIYWxsdWNpbmF0ZWQgc3RyZWFtcyBpbmN1ciAtMSBkQiBwZXIgc3RyZWFtLgogIEFwcGxpZWQgYXQgUElUIHNlbGVjdGlvbiB0aW1lOiB0aGUgUElUIG1hdHJpeCBpcyBleHRlbmRlZCB3aXRoIHplcm8tY29sdW1ucwogIGZvciBtaXNzaW5nIHJlZmVyZW5jZXMgc28gdGhlIEh1bmdhcmlhbiBhc3NpZ25tZW50IGNhbiBtYXAgYSBzdHJlYW0gdG8gc2lsZW5jZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKCl9FUFMgPSAxZS0xMApfSEFMTF9QRU5BTFRZX0RCID0gLTEuMApfU1RGVF9XSU4gPSAxMjgKX1NURlRfSE9QID0gNjQKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIFNJLVNOUgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKCmRlZiBzaV9zbnIoZXN0aW1hdGU6IHRvcmNoLlRlbnNvciwgdGFyZ2V0OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICIiIgogICAgU2NhbGUtaW52YXJpYW50IFNOUiwgc2hhcGUtYWdub3N0aWMuCgogICAgQXJnczoKICAgICAgICBlc3RpbWF0ZTogWy4uLiwgVF9lXQogICAgICAgIHRhcmdldDogICBbLi4uLCBUX3JdICAobWF5IGRpZmZlciBmcm9tIFRfZSBkdWUgdG8gU1RGVC9pU1RGVCBib3VuZGFyeSkKICAgIFJldHVybnM6CiAgICAgICAgWy4uLl0gU0ktU05SIGluIGRCLgogICAgIiIiCiAgICBtaW5fdCA9IG1pbihlc3RpbWF0ZS5zaGFwZVstMV0sIHRhcmdldC5zaGFwZVstMV0pCiAgICBlc3RpbWF0ZSA9IGVzdGltYXRlWy4uLiwgOm1pbl90XQogICAgdGFyZ2V0ID0gdGFyZ2V0Wy4uLiwgOm1pbl90XQogICAgZXN0aW1hdGUgPSBlc3RpbWF0ZSAtIGVzdGltYXRlLm1lYW4oZGltPS0xLCBrZWVwZGltPVRydWUpCiAgICB0YXJnZXQgPSB0YXJnZXQgLSB0YXJnZXQubWVhbihkaW09LTEsIGtlZXBkaW09VHJ1ZSkKICAgIGRvdCA9IChlc3RpbWF0ZSAqIHRhcmdldCkuc3VtKGRpbT0tMSwga2VlcGRpbT1UcnVlKQogICAgc190YXJnZXQgPSBkb3QgLyAodGFyZ2V0LnBvdygyKS5zdW0oZGltPS0xLCBrZWVwZGltPVRydWUpICsgX0VQUykgKiB0YXJnZXQKICAgIG5vaXNlID0gZXN0aW1hdGUgLSBzX3RhcmdldAogICAgcmV0dXJuIDEwLjAgKiB0b3JjaC5sb2cxMChzX3RhcmdldC5wb3coMikuc3VtKC0xKSAvIChub2lzZS5wb3coMikuc3VtKC0xKSArIF9FUFMpICsgX0VQUykKCgpkZWYgcGl0X3NpX3Nucihlc3RpbWF0ZXM6IHRvcmNoLlRlbnNvciwgcmVmZXJlbmNlczogdG9yY2guVGVuc29yKSAtPiB0dXBsZVt0b3JjaC5UZW5zb3IsIGxpc3RbbGlzdFtpbnRdXV06CiAgICAiIiIKICAgIFBlcm11dGF0aW9uLUludmFyaWFudCBUcmFpbmluZyBTSS1TTlIuCgogICAgQXJnczoKICAgICAgICBlc3RpbWF0ZXM6ICAoQiwgS19oYXQsIFQpIHNlcGFyYXRlZCBzdHJlYW1zLgogICAgICAgIHJlZmVyZW5jZXM6IChCLCBLX3JlZiwgVCkgY2xlYW4gcmVmZXJlbmNlcy4KICAgIFJldHVybnM6CiAgICAgICAgbWVhbl9zaXNucjogKEIsKSBtZWFuIFNJLVNOUiBhZnRlciBvcHRpbWFsIHBlcm11dGF0aW9uLgogICAgICAgIHBlcm1zOiBMaXN0IG9mIEIgcGVybXV0YXRpb24gbGlzdHMgbWFwcGluZyBLX2hhdCDihpIgS19yZWYuCiAgICAiIiIKICAgIEIsIEtfaGF0LCBUX2VzdCA9IGVzdGltYXRlcy5zaGFwZQogICAgS19yZWYsIFRfcmVmID0gcmVmZXJlbmNlcy5zaGFwZVsxXSwgcmVmZXJlbmNlcy5zaGFwZVsyXQogICAgVCA9IG1pbihUX2VzdCwgVF9yZWYpICAjIGFsaWduIGxlbmd0aHM6IGlTVEZUIGNhbiBkaWZmZXIgYnkgYSBmZXcgc2FtcGxlcwogICAgZXN0aW1hdGVzID0gZXN0aW1hdGVzWy4uLiwgOlRdCiAgICByZWZlcmVuY2VzID0gcmVmZXJlbmNlc1suLi4sIDpUXQogICAgSyA9IG1heChLX2hhdCwgS19yZWYpCgogICAgIyBQYWQgc21hbGxlciB0ZW5zb3IgdG8gSy4KICAgIGlmIEtfaGF0IDwgSzoKICAgICAgICBwYWQgPSB0b3JjaC56ZXJvcyhCLCBLIC0gS19oYXQsIFQsIGRldmljZT1lc3RpbWF0ZXMuZGV2aWNlKQogICAgICAgIGVzdGltYXRlc19wYWRkZWQgPSB0b3JjaC5jYXQoW2VzdGltYXRlcywgcGFkXSwgZGltPTEpCiAgICBlbHNlOgogICAgICAgIGVzdGltYXRlc19wYWRkZWQgPSBlc3RpbWF0ZXMKCiAgICBpZiBLX3JlZiA8IEs6CiAgICAgICAgcGFkID0gdG9yY2guemVyb3MoQiwgSyAtIEtfcmVmLCBULCBkZXZpY2U9cmVmZXJlbmNlcy5kZXZpY2UpCiAgICAgICAgcmVmc19wYWRkZWQgPSB0b3JjaC5jYXQoW3JlZmVyZW5jZXMsIHBhZF0sIGRpbT0xKQogICAgZWxzZToKICAgICAgICByZWZzX3BhZGRlZCA9IHJlZmVyZW5jZXMKCiAgICAjIEJ1aWxkIGNvc3QgbWF0cml4IFtCLCBLLCBLXS4KICAgIGNvc3QgPSB0b3JjaC56ZXJvcyhCLCBLLCBLLCBkZXZpY2U9ZXN0aW1hdGVzLmRldmljZSkKICAgIGZvciBpIGluIHJhbmdlKEspOgogICAgICAgIGZvciBqIGluIHJhbmdlKEspOgogICAgICAgICAgICBjb3N0WzosIGksIGpdID0gLXNpX3Nucihlc3RpbWF0ZXNfcGFkZGVkWzosIGldLCByZWZzX3BhZGRlZFs6LCBqXSkKCiAgICAjIEh1bmdhcmlhbiB2aWEgc2NpcHkgKENQVSBvbmx5IGZvciBub3cg4oCUIHNtYWxsIEspLgogICAgIyBJTVBPUlRBTlQ6IHVzZSAtY29zdCB2YWx1ZXMgZnJvbSB0aGUgcHJlLWNvbXB1dGVkIFB5VG9yY2ggY29zdCBtYXRyaXgKICAgICMgcmF0aGVyIHRoYW4gY2FsbGluZyBzaV9zbnIoKSBhZ2Fpbiwgc28gZ3JhZGllbnQgZmxvd3MgdGhyb3VnaCBjb3N0IOKGkiBlc3RpbWF0ZXMuCiAgICBmcm9tIHNjaXB5Lm9wdGltaXplIGltcG9ydCBsaW5lYXJfc3VtX2Fzc2lnbm1lbnQKICAgIHBlcm1zOiBsaXN0W2xpc3RbaW50XV0gPSBbXQogICAgc2lzbnJfcGVyX3NhbXBsZTogbGlzdFt0b3JjaC5UZW5zb3JdID0gW10KCiAgICBmb3IgYiBpbiByYW5nZShCKToKICAgICAgICByb3dfaW5kLCBjb2xfaW5kID0gbGluZWFyX3N1bV9hc3NpZ25tZW50KGNvc3RbYl0uZGV0YWNoKCkuY3B1KCkubnVtcHkoKSkKICAgICAgICBwZXJtID0gbGlzdChjb2xfaW5kWzpLX2hhdF0pCiAgICAgICAgcGVybXMuYXBwZW5kKHBlcm0pCiAgICAgICAgbl9wYWlycyA9IG1pbihLX2hhdCwgS19yZWYpCiAgICAgICAgIyBLZWVwIHZhbHVlcyBhcyB0ZW5zb3JzIChOT1QgLml0ZW0oKSkgc28gYmFja3dhcmQgY2FuIGZsb3cgdGhyb3VnaCB0aGVtLgogICAgICAgIG1hdGNoZWQgPSB0b3JjaC5zdGFjayhbLWNvc3RbYiwgcm93X2luZFtpXSwgY29sX2luZFtpXV0gZm9yIGkgaW4gcmFuZ2Uobl9wYWlycyldKQogICAgICAgIG5faGFsbCA9IG1heCgwLCBLX2hhdCAtIEtfcmVmKQogICAgICAgIHNpc25yX3Blcl9zYW1wbGUuYXBwZW5kKG1hdGNoZWQubWVhbigpICsgbl9oYWxsICogX0hBTExfUEVOQUxUWV9EQikKCiAgICBzaXNucl92YWxzID0gdG9yY2guc3RhY2soc2lzbnJfcGVyX3NhbXBsZSkgICMgKEIsKSB3aXRoIGdyYWRfZm4KICAgIHJldHVybiBzaXNucl92YWxzLCBwZXJtcwoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgU1RGVCBtYWduaXR1ZGUgbG9zcwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKCmRlZiBfbWFnX3N0ZnQod2F2ZWZvcm06IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgIiIiW0IsIFRdIOKGkiBbQiwgRiwgZnJhbWVzXSBtYWduaXR1ZGUgU1RGVCBhdCA4IGtIeiBTUi1Db3JyTmV0IHBhcmFtcy4iIiIKICAgIEIsIFQgPSB3YXZlZm9ybS5zaGFwZQogICAgd2luZG93ID0gdG9yY2guaGFubl93aW5kb3coX1NURlRfV0lOLCBkZXZpY2U9d2F2ZWZvcm0uZGV2aWNlKQogICAgc3BlYyA9IHRvcmNoLnN0ZnQoCiAgICAgICAgd2F2ZWZvcm0sCiAgICAgICAgbl9mZnQ9X1NURlRfV0lOLAogICAgICAgIGhvcF9sZW5ndGg9X1NURlRfSE9QLAogICAgICAgIHdpbl9sZW5ndGg9X1NURlRfV0lOLAogICAgICAgIHdpbmRvdz13aW5kb3csCiAgICAgICAgcmV0dXJuX2NvbXBsZXg9VHJ1ZSwKICAgICAgICBjZW50ZXI9VHJ1ZSwKICAgICkKICAgIHJldHVybiBzcGVjLmFicygpICAjIFtCLCBGLCBmcmFtZXNdCgoKZGVmIHBpdF9zaV9zbnJfbWFnKGVzdGltYXRlczogdG9yY2guVGVuc29yLCByZWZlcmVuY2VzOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICIiIlBJVCBTSS1TTlIgb24gbWFnbml0dWRlIFNURlQsIGF2ZXJhZ2VkIG92ZXIgQi4gUmV0dXJucyBzY2FsYXIuIiIiCiAgICBCLCBLX2hhdCwgVCA9IGVzdGltYXRlcy5zaGFwZQogICAgS19yZWYgPSByZWZlcmVuY2VzLnNoYXBlWzFdCgogICAgIyBGbGF0dGVuIHRvIFtCKkssIFRdIGZvciBiYXRjaCBTVEZULgogICAgZXN0X21hZyA9IF9tYWdfc3RmdChlc3RpbWF0ZXMucmVzaGFwZShCICogS19oYXQsIFQpKS5yZXNoYXBlKEIsIEtfaGF0LCAtMSkKICAgIHJlZl9tYWcgPSBfbWFnX3N0ZnQocmVmZXJlbmNlcy5yZXNoYXBlKEIgKiBLX3JlZiwgVCkpLnJlc2hhcGUoQiwgS19yZWYsIC0xKQoKICAgIHNpc25yX3ZhbHMsIF8gPSBwaXRfc2lfc25yKGVzdF9tYWcsIHJlZl9tYWcpCiAgICByZXR1cm4gc2lzbnJfdmFscy5tZWFuKCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIEF0dHJhY3RvciBCQ0UKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCgpkZWYgYXR0cmFjdG9yX2JjZShsb2dpdHM6IHRvcmNoLlRlbnNvciwgbl9zcGVha2VyczogbGlzdFtpbnRdKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAiIiIKICAgIEJDRSBvbiBhdHRyYWN0b3IgcHJlc2VuY2UgbG9naXRzIChzaGFwZSBCLCA3IG9yIDEsIDcpLgoKICAgIEFyZ3M6CiAgICAgICAgbG9naXRzOiBSYXcgbG9naXRzIGZyb20gcHJlc1sibG9naXRzIl0sIHNoYXBlIChCLCA3KSBvciAoQiwgMSwgNykuCiAgICAgICAgbl9zcGVha2VyczogVHJ1ZSBzcGVha2VyIGNvdW50IHBlciBzYW1wbGUuCiAgICBSZXR1cm5zOgogICAgICAgIFNjYWxhciBCQ0UgbG9zcy4KICAgICIiIgogICAgbG9naXRzID0gbG9naXRzLnNxdWVlemUoMSkgaWYgbG9naXRzLm5kaW0gPT0gMyBlbHNlIGxvZ2l0cyAgIyAoQiwgNykKICAgIEIgPSBsb2dpdHMuc2hhcGVbMF0KICAgIHRhcmdldHMgPSB0b3JjaC56ZXJvc19saWtlKGxvZ2l0cykKICAgIGZvciBiLCBuIGluIGVudW1lcmF0ZShuX3NwZWFrZXJzKToKICAgICAgICAjIFNsb3RzIDEuLm4gYXJlIGFjdGl2ZS4KICAgICAgICB0YXJnZXRzW2IsIDEgOiBuICsgMV0gPSAxLjAKICAgIHJldHVybiBGLmJpbmFyeV9jcm9zc19lbnRyb3B5X3dpdGhfbG9naXRzKGxvZ2l0cywgdGFyZ2V0cykKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIENvbWJpbmVkIHRyYWluaW5nIGxvc3MKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCgpkZWYgY2FsbXNlcF9sb3NzKAogICAgZXN0aW1hdGVzOiB0b3JjaC5UZW5zb3IsCiAgICByZWZlcmVuY2VzOiB0b3JjaC5UZW5zb3IsCiAgICBsb2dpdHM6IHRvcmNoLlRlbnNvciB8IE5vbmUsCiAgICBuX3NwZWFrZXJzOiBsaXN0W2ludF0sCiAgICBtYWdfd2VpZ2h0OiBmbG9hdCA9IDAuNSwKICAgIGF0dF93ZWlnaHQ6IGZsb2F0ID0gMC4xLAopIC0+IGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdOgogICAgIiIiCiAgICBDb21iaW5lZCBDQUxNLVNlcCBzZXBhcmF0aW9uIGxvc3MuCgogICAgPSBQSVRfU0lTTlJfdGltZSArIG1hZ193ZWlnaHQgw5cgUElUX1NJU05SX21hZyArIGF0dF93ZWlnaHQgw5cgQkNFKGF0dHJhY3RvcikKCiAgICBBcmdzOgogICAgICAgIGVzdGltYXRlczogIChCLCBLX2hhdCwgVCkgc2VwYXJhdGVkIHdhdmVmb3Jtcy4KICAgICAgICByZWZlcmVuY2VzOiAoQiwgS19yZWYsIFQpIGNsZWFuIHJlZmVyZW5jZSB3YXZlZm9ybXMuCiAgICAgICAgbG9naXRzOiAgICAgKEIsIDcpIG9yIE5vbmUuIElmIE5vbmUsIGF0dHJhY3RvciBsb3NzIGlzIHNraXBwZWQuCiAgICAgICAgbl9zcGVha2VyczogVHJ1ZSBjb3VudCBwZXIgc2FtcGxlLgogICAgICAgIG1hZ193ZWlnaHQ6IFdlaWdodCBvbiBTVEZUIG1hZ25pdHVkZSBsb3NzLgogICAgICAgIGF0dF93ZWlnaHQ6IFdlaWdodCBvbiBhdHRyYWN0b3IgQkNFLgoKICAgIFJldHVybnM6CiAgICAgICAgRGljdCB3aXRoICd0b3RhbCcsICd0aW1lJywgJ21hZycsICdhdHQnIHNjYWxhciB0ZW5zb3JzLgogICAgIiIiCiAgICAjIEFsaWduIHRpbWUgYXhpczogaVNURlQgbWF5IHByb2R1Y2UgwrFmZXcgc2FtcGxlcyB2cyB0aGUgcmVmZXJlbmNlCiAgICBtaW5fdCA9IG1pbihlc3RpbWF0ZXMuc2hhcGVbLTFdLCByZWZlcmVuY2VzLnNoYXBlWy0xXSkKICAgIGVzdGltYXRlcyA9IGVzdGltYXRlc1suLi4sIDptaW5fdF0KICAgIHJlZmVyZW5jZXMgPSByZWZlcmVuY2VzWy4uLiwgOm1pbl90XQoKICAgIHNpc25yX3ZhbHMsIF8gPSBwaXRfc2lfc25yKGVzdGltYXRlcywgcmVmZXJlbmNlcykKICAgIGxvc3NfdGltZSA9IC1zaXNucl92YWxzLm1lYW4oKQogICAgbG9zc19tYWcgPSAtcGl0X3NpX3Nucl9tYWcoZXN0aW1hdGVzLCByZWZlcmVuY2VzKQoKICAgIHRvdGFsID0gbG9zc190aW1lICsgbWFnX3dlaWdodCAqIGxvc3NfbWFnCgogICAgbG9zc19hdHQgPSB0b3JjaC50ZW5zb3IoMC4wLCBkZXZpY2U9ZXN0aW1hdGVzLmRldmljZSkKICAgIGlmIGxvZ2l0cyBpcyBub3QgTm9uZToKICAgICAgICBsb3NzX2F0dCA9IGF0dHJhY3Rvcl9iY2UobG9naXRzLCBuX3NwZWFrZXJzKQogICAgICAgIHRvdGFsID0gdG90YWwgKyBhdHRfd2VpZ2h0ICogbG9zc19hdHQKCiAgICByZXR1cm4geyJ0b3RhbCI6IHRvdGFsLCAidGltZSI6IGxvc3NfdGltZSwgIm1hZyI6IGxvc3NfbWFnLCAiYXR0IjogbG9zc19hdHR9Cg=='))
os.makedirs(f'{PROJ}/models', exist_ok=True)
open(f'{PROJ}/models/__init__.py', 'wb').write(base64.b64decode('IiIiRXhwZXJ0IG1vZGVscywgY2FzY2FkZSBnYXRlLCBmdXNpb24gaGVhZCAoRGV2IEIpLiIiIgo='))
os.makedirs(f'{PROJ}/models', exist_ok=True)
open(f'{PROJ}/models/lora.py', 'wb').write(base64.b64decode('...'))
os.makedirs(f'{PROJ}/data', exist_ok=True)
open(f'{PROJ}/data/__init__.py', 'wb').write(base64.b64decode('IiIiRGF0YSBwaXBlbGluZTogZHluYW1pYyBtaXhlciwgYXVnbWVudGF0aW9uLCBhbmQgZGF0YXNldCBwcmVwYXJhdGlvbiAoRGV2IEEpLiIiIgo='))
os.makedirs(f'{PROJ}/data', exist_ok=True)
open(f'{PROJ}/data/calmsep_mixer.py', 'wb').write(base64.b64decode('IiIiCkNBTE0tU2VwIDgga0h6IGR5bmFtaWMgbWl4ZXIgd2l0aCBkZWdyYWRhdGlvbiByZWNpcGUgbG9nZ2luZyAoRGV2IEEsIFAwLUExKS4KClRoZSBmcm96ZW4gU1ItQ29yck5ldCB2YXItMi01IGNoZWNrcG9pbnQgb3BlcmF0ZXMgYXQgOCBrSHogKFNURlQgd2luZG93IDEyOCwKaG9wIDY0KS4gRXZlcnkgdHJhaW5pbmcgYW5kIGV2YWx1YXRpb24gbWl4dHVyZSBpbiBDQUxNLVNlcCBpcyB0aGVyZWZvcmUgbWl4ZWQKYXQgOCBrSHouIFRoaXMgbW9kdWxlIGlzIHRoZSA4IGtIeiBjb3VudGVycGFydCB0byBkYXRhL21peGVyLnB5LCB3aGljaCBzdGF5cwphdCAxNiBrSHogZm9yIHRoZSBsZWdhY3kgMTYga0h6IHBhdGggYW5kIHRoZSBiYW5kLXJlY292ZXJ5IHRhcmdldHMuCgpUaGUgY3JpdGljYWwgZGlmZmVyZW5jZSBmcm9tIGRhdGEvbWl4ZXIucHkgaXMgdGhlIHJlY2lwZSBsb2cuIEJMVUVQUklOVCBzZWN0aW9uCjUuNCByZXF1aXJlcyB0aGF0IGV2ZXJ5IGNvbmRpdGlvbiBsYWJlbCAoU05SLCBUNjAsIGNvZGVjIGZhbWlseSBhbmQgYml0cmF0ZSwKc3BlYWtlciBjb3VudCkgY29tZSBmcmVlIGZyb20gdGhlIHN5bnRoZXNpcyByZWNpcGUgcmF0aGVyIHRoYW4gZnJvbSBhIG5ldXJhbAplc3RpbWF0ZS4gVGhpcyBtaXhlciByZXR1cm5zIGEgTWl4dHVyZVJlY2lwZSBhbG9uZ3NpZGUgZXZlcnkgbWl4dHVyZSByZWNvcmRpbmcKZXhhY3RseSB3aGF0IHdhcyBhcHBsaWVkLCBzbyB0aGUgY29uZGl0aW9uIGFuYWx5emVyIGFuZCBnYXRlIHRyYWluIGFnYWluc3QKZ3JvdW5kIHRydXRoIHRoYXQgd2FzIG5ldmVyIGVzdGltYXRlZC4KCkRlZ3JhZGF0aW9ucyBhcmUgYXBwbGllZCBieSBkYXRhL2RlZ3JhZGF0aW9ucy5weTsgdGhpcyBtb2R1bGUgb3ducyB0aGUgc291cmNlCmRyYXcsIGxldmVsIG9mZnNldHMsIHN1bW1hdGlvbiwgYW5kIHRoZSByZWNpcGUgcmVjb3JkLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB1dWlkCmZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBTZXF1ZW5jZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBhc2RpY3QsIGRhdGFjbGFzcywgZmllbGQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSBkYXRhLm1peGVyX3N0dWIgaW1wb3J0IE1peHR1cmVTYW1wbGUsIF9sb2FkX3dhdgoKQ0FMTVNFUF9TQU1QTEVfUkFURTogaW50ID0gOF8wMDAKIiIiTG9ja2VkIGJ5IHRoZSBmcm96ZW4gY2hlY2twb2ludC4gQkxVRVBSSU5UIGZpeGVkIGNvbnN0cmFpbnRzLiBOZXZlciBjaGFuZ2UuIiIiCgpCQU5EX1JFQ09WRVJZX1NBTVBMRV9SQVRFOiBpbnQgPSAxNl8wMDAKIiIiUmF0ZSBmb3IgYmFuZC1yZWNvdmVyeSB0YXJnZXRzIGFuZCBETlNNT1Mgc2NvcmluZyBvbmx5LiBOZXZlciBmZWQgdG8gdGhlIGJhc2UuIiIiCgpfREVGQVVMVF9BTExPV0VEX046IGxpc3RbaW50XSA9IFsyLCAzLCA0LCA1XQoiIiJOIGluIHsyLDMsNCw1fS4gSzA9NSBpbiB0aGUgY2hlY2twb2ludDsgdGhlcmUgaXMgbm8gNisgc3BlYWtlciByZWdpbWUuIiIiCgoKQGRhdGFjbGFzcwpjbGFzcyBNaXh0dXJlUmVjaXBlOgogICAgIiIiCiAgICBHcm91bmQtdHJ1dGggcmVjb3JkIG9mIGV2ZXJ5dGhpbmcgYXBwbGllZCB0byBvbmUgbWl4dHVyZS4KCiAgICBFdmVyeSBmaWVsZCBoZXJlIGlzIGEgZnJlZSBzdXBlcnZpc2lvbiB0YXJnZXQ6IGl0IGlzIGtub3duIGJlY2F1c2UgdGhpcwogICAgY29kZSBjaG9zZSBpdCwgbm90IGJlY2F1c2UgYSBtb2RlbCBlc3RpbWF0ZWQgaXQuIFRoZSBjb25kaXRpb24gYW5hbHl6ZXIKICAgIChCTFVFUFJJTlQgNS40KSBhbmQgdGhlIGdhdGUgKDUuNSkgdHJhaW4gYWdhaW5zdCB0aGVzZSB2YWx1ZXMuCgogICAgQXR0cmlidXRlczoKICAgICAgICBuX3NwZWFrZXJzOiBUcnVlIHNwZWFrZXIgY291bnQsIHRoZSBwcmltYXJ5IGNvdW50aW5nIGxhYmVsLgogICAgICAgIHNwZWFrZXJfaWRzOiBTb3VyY2Ugc3BlYWtlciBJRHMsIGluIHJlZmVyZW5jZS1zdHJlYW0gb3JkZXIuCiAgICAgICAgc291cmNlX2ZpbGVzOiBTb3VyY2UgdXR0ZXJhbmNlIHBhdGhzLCBpbiByZWZlcmVuY2Utc3RyZWFtIG9yZGVyLgogICAgICAgIGxldmVsX29mZnNldHNfZGI6IFBlci1zcGVha2VyIGdhaW4gYXBwbGllZCwgaW4gcmVmZXJlbmNlLXN0cmVhbSBvcmRlci4KICAgICAgICBzbnJfZGI6IE5vaXNlIFNOUiBpbiBkQiwgb3IgTm9uZSB3aGVuIG5vIG5vaXNlIHdhcyBhZGRlZC4KICAgICAgICBub2lzZV9maWxlOiBOb2lzZSBzb3VyY2UgcGF0aCwgb3IgTm9uZS4KICAgICAgICB0NjBfczogUmV2ZXJiZXJhdGlvbiB0aW1lIGluIHNlY29uZHMsIG9yIE5vbmUgd2hlbiBhbmVjaG9pYy4KICAgICAgICByaXJfZmlsZTogUklSIHBhdGggdXNlZCwgb3IgTm9uZS4KICAgICAgICBjb2RlY19uYW1lOiBDb2RlYyBmYW1pbHkgYXBwbGllZCAoIm9wdXMiLCAiYWFjIiwgImFtci1uYiIsICJhbXItd2IiKSwKICAgICAgICAgICAgb3IgTm9uZSB3aGVuIHVuY29tcHJlc3NlZC4KICAgICAgICBjb2RlY19iaXRyYXRlX2JwczogQ29kZWMgYml0cmF0ZSBpbiBiaXRzL3NlYywgb3IgTm9uZS4KICAgICAgICBzZWVkOiBSTkcgc2VlZCB0aGF0IHByb2R1Y2VkIHRoaXMgbWl4dHVyZSwgd2hlbiB0aGUgbWl4ZXIgd2FzIHNlZWRlZC4KICAgICAgICBzYW1wbGVfcmF0ZTogQWx3YXlzIENBTE1TRVBfU0FNUExFX1JBVEUuCiAgICAiIiIKCiAgICBuX3NwZWFrZXJzOiBpbnQKICAgIHNwZWFrZXJfaWRzOiBsaXN0W3N0cl0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHNvdXJjZV9maWxlczogbGlzdFtzdHJdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBsZXZlbF9vZmZzZXRzX2RiOiBsaXN0W2Zsb2F0XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgc25yX2RiOiBmbG9hdCB8IE5vbmUgPSBOb25lCiAgICBub2lzZV9maWxlOiBzdHIgfCBOb25lID0gTm9uZQogICAgdDYwX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUKICAgIHJpcl9maWxlOiBzdHIgfCBOb25lID0gTm9uZQogICAgY29kZWNfbmFtZTogc3RyIHwgTm9uZSA9IE5vbmUKICAgIGNvZGVjX2JpdHJhdGVfYnBzOiBpbnQgfCBOb25lID0gTm9uZQogICAgc2VlZDogaW50IHwgTm9uZSA9IE5vbmUKICAgIHNhbXBsZV9yYXRlOiBpbnQgPSBDQUxNU0VQX1NBTVBMRV9SQVRFCgogICAgZGVmIHRvX2RpY3Qoc2VsZikgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgIiIiU2VyaWFsaXplIGZvciBtYW5pZmVzdCB3cml0aW5nIGFuZCBjb25kaXRpb24tbGFiZWwgZXh0cmFjdGlvbi4iIiIKICAgICAgICByZXR1cm4gYXNkaWN0KHNlbGYpCgogICAgZGVmIGNvbmRpdGlvbl92ZWN0b3Ioc2VsZikgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgICAgICAiIiIKICAgICAgICBUaGUgc3VwZXJ2aXNlZCBjb25kaXRpb24gdGFyZ2V0cywgYXMgdGhlIGFuYWx5emVyIGNvbnN1bWVzIHRoZW0uCgogICAgICAgIEFic2VudCBjb25kaXRpb25zIG1hcCB0byB0aGVpciBuZXV0cmFsIHZhbHVlIHJhdGhlciB0aGFuIE5vbmUgc28gdGhlCiAgICAgICAgdmVjdG9yIGlzIGFsd2F5cyBkZW5zZTogbm8gbm9pc2UgbWVhbnMgYSBoaWdoIFNOUiwgYW5lY2hvaWMgbWVhbnMgYQogICAgICAgIG5lYXItemVybyBUNjAsIHVuY29tcHJlc3NlZCBtZWFucyBjb2RlYyBjbGFzcyAwLiBUaGlzIGlzIHdoYXQgbWFrZXMKICAgICAgICB0aGUgZ2F0ZSdzIGNsZWFuLWlucHV0IHRhcmdldCAoYWxsIGdhdGVzIG5lYXIgemVybykgd2VsbCBkZWZpbmVkLgoKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBEaWN0IHdpdGgga2V5cyBzbnJfZGIsIHQ2MF9zLCBjb2RlY19jbGFzcywgY29kZWNfYml0cmF0ZV9rYnBzLAogICAgICAgICAgICBuX3NwZWFrZXJzLiBDb25zdW1lZCBieSBtb2RlbHMvY29uZGl0aW9uLnB5LgogICAgICAgICIiIgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJzbnJfZGIiOiA2MC4wIGlmIHNlbGYuc25yX2RiIGlzIE5vbmUgZWxzZSBmbG9hdChzZWxmLnNucl9kYiksCiAgICAgICAgICAgICJ0NjBfcyI6IDAuMCBpZiBzZWxmLnQ2MF9zIGlzIE5vbmUgZWxzZSBmbG9hdChzZWxmLnQ2MF9zKSwKICAgICAgICAgICAgImNvZGVjX2NsYXNzIjogZmxvYXQoX0NPREVDX0NMQVNTX0lOREVYLmdldChzZWxmLmNvZGVjX25hbWUgb3IgIm5vbmUiLCAwKSksCiAgICAgICAgICAgICJjb2RlY19iaXRyYXRlX2ticHMiOiAoCiAgICAgICAgICAgICAgICAwLjAgaWYgc2VsZi5jb2RlY19iaXRyYXRlX2JwcyBpcyBOb25lIGVsc2Ugc2VsZi5jb2RlY19iaXRyYXRlX2JwcyAvIDEwMDAuMAogICAgICAgICAgICApLAogICAgICAgICAgICAibl9zcGVha2VycyI6IGZsb2F0KHNlbGYubl9zcGVha2VycyksCiAgICAgICAgfQoKCl9DT0RFQ19DTEFTU19JTkRFWDogZGljdFtzdHIsIGludF0gPSB7CiAgICAibm9uZSI6IDAsCiAgICAib3B1cyI6IDEsCiAgICAiYWFjIjogMiwKICAgICJhbXItbmIiOiAzLAogICAgImFtci13YiI6IDQsCn0KIiIiQ29kZWMgZmFtaWx5IHRvIGNsYXNzIGluZGV4LiBJbmRleCAwIChub25lKSBpcyB0aGUgY2xlYW4vbmV1dHJhbCBjbGFzcy4iIiIKCgpAZGF0YWNsYXNzCmNsYXNzIENhbG1TZXBNaXh0dXJlOgogICAgIiIiCiAgICBPbmUgOCBrSHogbWl4dHVyZSB3aXRoIGl0cyBzdGVtcyBhbmQgaXRzIGdyb3VuZC10cnV0aCByZWNpcGUuCgogICAgQXR0cmlidXRlczoKICAgICAgICBzYW1wbGU6IFRoZSBtaXh0dXJlIGFuZCByZWZlcmVuY2Ugc3RlbXMgKE1peHR1cmVTYW1wbGUsIDgga0h6KS4KICAgICAgICByZWNpcGU6IFdoYXQgd2FzIGFwcGxpZWQsIGZvciBzdXBlcnZpc2lvbiBhbmQgbWFuaWZlc3RzLgogICAgIiIiCgogICAgc2FtcGxlOiBNaXh0dXJlU2FtcGxlCiAgICByZWNpcGU6IE1peHR1cmVSZWNpcGUKCiAgICBAcHJvcGVydHkKICAgIGRlZiBtaXh0dXJlKHNlbGYpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgcmV0dXJuIHNlbGYuc2FtcGxlLm1peHR1cmUKCiAgICBAcHJvcGVydHkKICAgIGRlZiByZWZlcmVuY2VzKHNlbGYpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgcmV0dXJuIHNlbGYuc2FtcGxlLnJlZmVyZW5jZXMKCiAgICBAcHJvcGVydHkKICAgIGRlZiBuX3NwZWFrZXJzKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gc2VsZi5yZWNpcGUubl9zcGVha2VycwoKCmRlZiBfc3BlYWtlcl9pZChwYXRoOiBQYXRoKSAtPiBzdHI6CiAgICAiIiIKICAgIFNwZWFrZXIgSUQgZnJvbSBhIExpYnJpU3BlZWNoLXN0eWxlIGZpbGVuYW1lLgoKICAgIExpYnJpU3BlZWNoIG5hbWVzIGFyZSB7c3BlYWtlcn0te2NoYXB0ZXJ9LXt1dHRlcmFuY2V9LmV4dCwgc28gdGhlIElEIGlzIHRoZQogICAgY29tcG9uZW50IGJlZm9yZSB0aGUgZmlyc3QgZGFzaC4gTm9uLWNvbmZvcm1pbmcgbmFtZXMgZmFsbCBiYWNrIHRvIHRoZSBmdWxsCiAgICBzdGVtLCB3aGljaCBrZWVwcyBzcGVha2VyIGlzb2xhdGlvbiBjb25zZXJ2YXRpdmU6IGFuIHVucGFyc2VkIG5hbWUgaXMgaXRzCiAgICBvd24gc3BlYWtlciByYXRoZXIgdGhhbiBzaWxlbnRseSBjb2xsaWRpbmcgd2l0aCBhbm90aGVyLgogICAgIiIiCiAgICByZXR1cm4gcGF0aC5zdGVtLnNwbGl0KCItIilbMF0KCgpjbGFzcyBDYWxtU2VwTWl4ZXI6CiAgICAiIiIKICAgIERyYXdzIE4gY2xlYW4gOCBrSHogdXR0ZXJhbmNlcyBhbmQgbWl4ZXMgdGhlbSwgbG9nZ2luZyB0aGUgcmVjaXBlLgoKICAgIFNwZWFrZXIgaXNvbGF0aW9uIGlzIGVuZm9yY2VkIGF0IGNvbnN0cnVjdGlvbjogZmlsZXMgYmVsb25naW5nIHRvIGhlbGQtb3V0CiAgICBzcGVha2VycyBhcmUgcmVtb3ZlZCBmcm9tIHRoZSB0cmFpbmluZyBwb29sIGVudGlyZWx5LCBzbyBhIGRldi1jbGVhbiBvcgogICAgdGVzdC1jbGVhbiBzcGVha2VyIGNhbiBuZXZlciBsZWFrIGludG8gdHJhaW5pbmcgKEJMVUVQUklOVCA3LjUsIGhvbGRvdXQgMSkuCgogICAgUGFyYW1ldGVycwogICAgLS0tLS0tLS0tLQogICAgc291cmNlX2ZpbGVzOgogICAgICAgIENsZWFuIHNpbmdsZS1zcGVha2VyIFdBVi9GTEFDIGZpbGVzLCBhbHJlYWR5IGF0IDgga0h6LgogICAgYWxsb3dlZF9uOgogICAgICAgIFNwZWFrZXIgY291bnRzIHRvIGRyYXcgZnJvbS4gRGVmYXVsdHMgdG8gWzIsIDMsIDQsIDVdLgogICAgZGJfbWluLCBkYl9tYXg6CiAgICAgICAgUGVyLXNwZWFrZXIgbGV2ZWwgb2Zmc2V0IHJhbmdlIGluIGRCLCBkcmF3biBpbmRlcGVuZGVudGx5IHBlciBzcGVha2VyLgogICAgaGVsZF9vdXRfc3BlYWtlcl9pZHM6CiAgICAgICAgU3BlYWtlcnMgcmVzZXJ2ZWQgZm9yIHZhbGlkYXRpb24gYW5kIGV2YWx1YXRpb24uIEV4Y2x1ZGVkIGZyb20gdGhlCiAgICAgICAgdHJhaW5pbmcgcG9vbCBhbmQgcmVhY2hhYmxlIG9ubHkgdmlhIGBgbWl4KHNwbGl0PSJoZWxkb3V0IilgYC4KICAgIHNhbXBsZV9yYXRlOgogICAgICAgIEV4cGVjdGVkIGlucHV0IHJhdGUuIERlZmF1bHRzIHRvIENBTE1TRVBfU0FNUExFX1JBVEUgKDgwMDApLiBBIGZpbGUgYXQKICAgICAgICBhbnkgb3RoZXIgcmF0ZSByYWlzZXMgcmF0aGVyIHRoYW4gYmVpbmcgc2lsZW50bHkgcmVzYW1wbGVkLCBiZWNhdXNlIGEKICAgICAgICBzaWxlbnQgcmVzYW1wbGUgaXMgaG93IGEgMTYga0h6IGZpbGUgZW5kcyB1cCBpbnRlcnByZXRlZCBhcyA4IGtIei4KICAgIHJuZzoKICAgICAgICBTZWVkZWQgZ2VuZXJhdG9yIGZvciByZXByb2R1Y2libGUgbWl4ZXMuCiAgICBzZWVkOgogICAgICAgIFJlY29yZGVkIGludG8gZXZlcnkgcmVjaXBlIGZvciB0cmFjZWFiaWxpdHkuIFBhc3MgdGhlIHNhbWUgdmFsdWUgdXNlZAogICAgICAgIHRvIGJ1aWxkIGBgcm5nYGAuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBzb3VyY2VfZmlsZXM6IFNlcXVlbmNlW1BhdGggfCBzdHJdLAogICAgICAgIGFsbG93ZWRfbjogbGlzdFtpbnRdIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgZGJfbWluOiBmbG9hdCA9IDAuMCwKICAgICAgICBkYl9tYXg6IGZsb2F0ID0gNS4wLAogICAgICAgIGhlbGRfb3V0X3NwZWFrZXJfaWRzOiBzZXRbc3RyXSB8IE5vbmUgPSBOb25lLAogICAgICAgIHNhbXBsZV9yYXRlOiBpbnQgPSBDQUxNU0VQX1NBTVBMRV9SQVRFLAogICAgICAgIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvciB8IE5vbmUgPSBOb25lLAogICAgICAgIHNlZWQ6IGludCB8IE5vbmUgPSBOb25lLAogICAgKSAtPiBOb25lOgogICAgICAgIGlmIGRiX21pbiA+IGRiX21heDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImRiX21pbiAoe2RiX21pbn0pIG11c3QgYmUgPD0gZGJfbWF4ICh7ZGJfbWF4fSkiKQoKICAgICAgICBzZWxmLl9hbGxvd2VkX24gPSBsaXN0KGFsbG93ZWRfbikgaWYgYWxsb3dlZF9uIGlzIG5vdCBOb25lIGVsc2UgbGlzdChfREVGQVVMVF9BTExPV0VEX04pCiAgICAgICAgaWYgbm90IHNlbGYuX2FsbG93ZWRfbjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYWxsb3dlZF9uIG11c3QgY29udGFpbiBhdCBsZWFzdCBvbmUgdmFsdWUiKQogICAgICAgIGJhZCA9IFtuIGZvciBuIGluIHNlbGYuX2FsbG93ZWRfbiBpZiBuIG5vdCBpbiAoMiwgMywgNCwgNSldCiAgICAgICAgaWYgYmFkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJhbGxvd2VkX24gY29udGFpbnMge2JhZH07IENBTE0tU2VwIHN1cHBvcnRzIE4gaW4ge3syLDMsNCw1fX0gb25seSAiCiAgICAgICAgICAgICAgICBmIihLMD01IGluIHRoZSBmcm96ZW4gY2hlY2twb2ludCkiCiAgICAgICAgICAgICkKCiAgICAgICAgc2VsZi5fZGJfbWluID0gZmxvYXQoZGJfbWluKQogICAgICAgIHNlbGYuX2RiX21heCA9IGZsb2F0KGRiX21heCkKICAgICAgICBzZWxmLl9zYW1wbGVfcmF0ZSA9IGludChzYW1wbGVfcmF0ZSkKICAgICAgICBzZWxmLl9ybmcgPSBybmcgaWYgcm5nIGlzIG5vdCBOb25lIGVsc2UgbnAucmFuZG9tLmRlZmF1bHRfcm5nKCkKICAgICAgICBzZWxmLl9zZWVkID0gc2VlZAoKICAgICAgICBhbGxfZmlsZXMgPSBbUGF0aChmKSBmb3IgZiBpbiBzb3VyY2VfZmlsZXNdCiAgICAgICAgaGVsZCA9IHNldChoZWxkX291dF9zcGVha2VyX2lkcykgaWYgaGVsZF9vdXRfc3BlYWtlcl9pZHMgZWxzZSBzZXQoKQoKICAgICAgICBzZWxmLl9oZWxkb3V0X2ZpbGVzOiBsaXN0W1BhdGhdID0gW2YgZm9yIGYgaW4gYWxsX2ZpbGVzIGlmIF9zcGVha2VyX2lkKGYpIGluIGhlbGRdCiAgICAgICAgc2VsZi5fdHJhaW5fZmlsZXM6IGxpc3RbUGF0aF0gPSBbZiBmb3IgZiBpbiBhbGxfZmlsZXMgaWYgX3NwZWFrZXJfaWQoZikgbm90IGluIGhlbGRdCgogICAgICAgIG1heF9uID0gbWF4KHNlbGYuX2FsbG93ZWRfbikKICAgICAgICBpZiBsZW4oc2VsZi5fdHJhaW5fZmlsZXMpIDwgbWF4X246CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICBmInRyYWluaW5nIHBvb2wgaGFzIHtsZW4oc2VsZi5fdHJhaW5fZmlsZXMpfSBmaWxlKHMpIGJ1dCAiCiAgICAgICAgICAgICAgICBmIm1heChhbGxvd2VkX24pPXttYXhfbn07IGFkZCBzb3VyY2VzIG9yIHJlZHVjZSBhbGxvd2VkX24iCiAgICAgICAgICAgICkKCiAgICAjIOKUgOKUgCBQdWJsaWMgQVBJIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiBtaXgoc2VsZiwgc3BsaXQ6IHN0ciA9ICJ0cmFpbiIsIG46IGludCB8IE5vbmUgPSBOb25lKSAtPiBDYWxtU2VwTWl4dHVyZToKICAgICAgICAiIiIKICAgICAgICBQcm9kdWNlIG9uZSBjbGVhbiA4IGtIeiBtaXh0dXJlIGFuZCBpdHMgcmVjaXBlLgoKICAgICAgICBEZWdyYWRhdGlvbnMgKHJldmVyYiwgbm9pc2UsIGNvZGVjKSBhcmUgYXBwbGllZCBhZnRlcndhcmRzIGJ5CiAgICAgICAgZGF0YS9kZWdyYWRhdGlvbnMucHksIHdoaWNoIGV4dGVuZHMgdGhlIHJldHVybmVkIHJlY2lwZSBpbiBwbGFjZS4gVGhpcwogICAgICAgIHNwbGl0IGtlZXBzIHRoZSBzb3VyY2UgZHJhdyBpbmRlcGVuZGVudCBvZiB0aGUgY29uZGl0aW9uIHNhbXBsaW5nLCBzbwogICAgICAgIHRoZSBzYW1lIG1peHR1cmUgY2FuIGJlIHJlbmRlcmVkIHVuZGVyIHNldmVyYWwgY29uZGl0aW9ucyBmb3IgdGhlCiAgICAgICAgbWF0Y2hlZC1wYWlyIGFuYWx5c2VzIGluIEJMVUVQUklOVCA5LjUuCgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHNwbGl0OiAidHJhaW4iIGRyYXdzIGZyb20gdGhlIHRyYWluaW5nIHBvb2wgKGhlbGQtb3V0IHNwZWFrZXJzCiAgICAgICAgICAgICAgICBleGNsdWRlZCksICJoZWxkb3V0IiBkcmF3cyBvbmx5IGZyb20gaGVsZC1vdXQgc3BlYWtlcnMsIGFueQogICAgICAgICAgICAgICAgb3RoZXIgdmFsdWUgZHJhd3MgZnJvbSBib3RoLgogICAgICAgICAgICBuOiBTcGVha2VyIGNvdW50IG92ZXJyaWRlLiBNdXN0IGJlIGluIGFsbG93ZWRfbi4gRHJhd24gdW5pZm9ybWx5CiAgICAgICAgICAgICAgICBmcm9tIGFsbG93ZWRfbiB3aGVuIE5vbmUuCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIENhbG1TZXBNaXh0dXJlIHdpdGggYW4gYW5lY2hvaWMsIG5vaXNlLWZyZWUsIHVuY29tcHJlc3NlZCBtaXh0dXJlCiAgICAgICAgICAgIGFuZCBhIHJlY2lwZSByZWNvcmRpbmcgdGhlIHNvdXJjZXMgYW5kIGxldmVsIG9mZnNldHMuCiAgICAgICAgIiIiCiAgICAgICAgY2hvc2VuX24gPSBzZWxmLl9yZXNvbHZlX24obikKICAgICAgICBwb29sID0gc2VsZi5fc2VsZWN0X3Bvb2woc3BsaXQpCgogICAgICAgIGlmIGxlbihwb29sKSA8IGNob3Nlbl9uOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJwb29sIGZvciBzcGxpdD17c3BsaXQhcn0gaGFzIHtsZW4ocG9vbCl9IGZpbGUocykgYnV0IG49e2Nob3Nlbl9ufSByZXF1ZXN0ZWQiCiAgICAgICAgICAgICkKCiAgICAgICAgaW5kaWNlcyA9IHNlbGYuX3JuZy5jaG9pY2UobGVuKHBvb2wpLCBzaXplPWNob3Nlbl9uLCByZXBsYWNlPUZhbHNlKQogICAgICAgIGNob3NlbiA9IFtwb29sW2ludChpKV0gZm9yIGkgaW4gaW5kaWNlc10KCiAgICAgICAgd2F2ZWZvcm1zOiBsaXN0W25wLm5kYXJyYXldID0gW10KICAgICAgICBvZmZzZXRzOiBsaXN0W2Zsb2F0XSA9IFtdCiAgICAgICAgZm9yIHBhdGggaW4gY2hvc2VuOgogICAgICAgICAgICBhdWRpbywgc3IgPSBfbG9hZF93YXYocGF0aCkKICAgICAgICAgICAgaWYgc3IgIT0gc2VsZi5fc2FtcGxlX3JhdGU6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgICAgIGYic2FtcGxlIHJhdGUgbWlzbWF0Y2g6IHtwYXRofSBpcyB7c3J9IEh6LCBleHBlY3RlZCB7c2VsZi5fc2FtcGxlX3JhdGV9IEh6LiAiCiAgICAgICAgICAgICAgICAgICAgZiJSZXNhbXBsZSB0aGUgY29ycHVzIHdpdGggZGF0YS9wcmVwYXJlX2xpYnJpc3BlZWNoXzhrLnB5IHJhdGhlciB0aGFuICIKICAgICAgICAgICAgICAgICAgICBmInJlc2FtcGxpbmcgaGVyZSwgc28gdGhlIHdob2xlIHBvb2wgc3RheXMgY29uc2lzdGVudC4iCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIGRiID0gZmxvYXQoc2VsZi5fcm5nLnVuaWZvcm0oc2VsZi5fZGJfbWluLCBzZWxmLl9kYl9tYXgpKQogICAgICAgICAgICBvZmZzZXRzLmFwcGVuZChkYikKICAgICAgICAgICAgd2F2ZWZvcm1zLmFwcGVuZCgoYXVkaW8gKiAoMTAuMCAqKiAoZGIgLyAyMC4wKSkpLmFzdHlwZShucC5mbG9hdDMyKSkKCiAgICAgICAgcmVmcywgbWl4dHVyZSA9IHNlbGYuX3BhZF9hbmRfc3VtKHdhdmVmb3JtcykKICAgICAgICB1aWQgPSBmImNhbG1zZXBfe2Nob3Nlbl9ufXNwa197dXVpZC51dWlkNCgpLmhleFs6OF19IgoKICAgICAgICByZWNpcGUgPSBNaXh0dXJlUmVjaXBlKAogICAgICAgICAgICBuX3NwZWFrZXJzPWNob3Nlbl9uLAogICAgICAgICAgICBzcGVha2VyX2lkcz1bX3NwZWFrZXJfaWQocCkgZm9yIHAgaW4gY2hvc2VuXSwKICAgICAgICAgICAgc291cmNlX2ZpbGVzPVtzdHIocCkgZm9yIHAgaW4gY2hvc2VuXSwKICAgICAgICAgICAgbGV2ZWxfb2Zmc2V0c19kYj1vZmZzZXRzLAogICAgICAgICAgICBzZWVkPXNlbGYuX3NlZWQsCiAgICAgICAgICAgIHNhbXBsZV9yYXRlPXNlbGYuX3NhbXBsZV9yYXRlLAogICAgICAgICkKICAgICAgICBzYW1wbGUgPSBNaXh0dXJlU2FtcGxlKAogICAgICAgICAgICBtaXh0dXJlPW1peHR1cmUsCiAgICAgICAgICAgIHJlZmVyZW5jZXM9cmVmcywKICAgICAgICAgICAgc2FtcGxlX3JhdGU9c2VsZi5fc2FtcGxlX3JhdGUsCiAgICAgICAgICAgIHV0dGVyYW5jZV9pZD11aWQsCiAgICAgICAgKQogICAgICAgIHJldHVybiBDYWxtU2VwTWl4dHVyZShzYW1wbGU9c2FtcGxlLCByZWNpcGU9cmVjaXBlKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHRyYWluX3Bvb2xfc2l6ZShzZWxmKSAtPiBpbnQ6CiAgICAgICAgIiIiTnVtYmVyIG9mIHNvdXJjZSBmaWxlcyBlbGlnaWJsZSBmb3IgdHJhaW5pbmcgbWl4ZXMuIiIiCiAgICAgICAgcmV0dXJuIGxlbihzZWxmLl90cmFpbl9maWxlcykKCiAgICBAcHJvcGVydHkKICAgIGRlZiBoZWxkb3V0X3Bvb2xfc2l6ZShzZWxmKSAtPiBpbnQ6CiAgICAgICAgIiIiTnVtYmVyIG9mIHNvdXJjZSBmaWxlcyByZXNlcnZlZCBmb3IgdmFsaWRhdGlvbiBhbmQgZXZhbHVhdGlvbi4iIiIKICAgICAgICByZXR1cm4gbGVuKHNlbGYuX2hlbGRvdXRfZmlsZXMpCgogICAgQHByb3BlcnR5CiAgICBkZWYgdHJhaW5fc3BlYWtlcnMoc2VsZikgLT4gc2V0W3N0cl06CiAgICAgICAgIiIiU3BlYWtlciBJRHMgcHJlc2VudCBpbiB0aGUgdHJhaW5pbmcgcG9vbC4iIiIKICAgICAgICByZXR1cm4ge19zcGVha2VyX2lkKGYpIGZvciBmIGluIHNlbGYuX3RyYWluX2ZpbGVzfQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGhlbGRvdXRfc3BlYWtlcnMoc2VsZikgLT4gc2V0W3N0cl06CiAgICAgICAgIiIiU3BlYWtlciBJRHMgcmVzZXJ2ZWQgZm9yIHZhbGlkYXRpb24gYW5kIGV2YWx1YXRpb24uIiIiCiAgICAgICAgcmV0dXJuIHtfc3BlYWtlcl9pZChmKSBmb3IgZiBpbiBzZWxmLl9oZWxkb3V0X2ZpbGVzfQoKICAgIGRlZiBhc3NlcnRfc3BlYWtlcl9pc29sYXRpb24oc2VsZikgLT4gTm9uZToKICAgICAgICAiIiIKICAgICAgICBQcm92ZSBubyBzcGVha2VyIGFwcGVhcnMgaW4gYm90aCBwb29scy4KCiAgICAgICAgQ2FsbGVkIGJ5IHRoZSBwcmVmbGlnaHQgY2hlY2sgYW5kIGJ5IHRlc3RzLiBBIHZpb2xhdGlvbiBoZXJlIG1lYW5zCiAgICAgICAgQkxVRVBSSU5UIGhvbGRvdXQgMSBpcyBicm9rZW4gYW5kIGV2ZXJ5IGRvd25zdHJlYW0gbnVtYmVyIGlzIHN1c3BlY3QsCiAgICAgICAgc28gdGhpcyByYWlzZXMgcmF0aGVyIHRoYW4gd2FybnMuCiAgICAgICAgIiIiCiAgICAgICAgb3ZlcmxhcCA9IHNlbGYudHJhaW5fc3BlYWtlcnMgJiBzZWxmLmhlbGRvdXRfc3BlYWtlcnMKICAgICAgICBpZiBvdmVybGFwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJzcGVha2VyIGlzb2xhdGlvbiB2aW9sYXRlZDoge3NvcnRlZChvdmVybGFwKX0gYXBwZWFyIGluIGJvdGggdGhlICIKICAgICAgICAgICAgICAgIGYidHJhaW5pbmcgYW5kIGhlbGQtb3V0IHBvb2xzIgogICAgICAgICAgICApCgogICAgIyDilIDilIAgUHJpdmF0ZSBoZWxwZXJzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiBfcmVzb2x2ZV9uKHNlbGYsIG46IGludCB8IE5vbmUpIC0+IGludDoKICAgICAgICBpZiBuIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBpbnQoc2VsZi5fcm5nLmNob2ljZShzZWxmLl9hbGxvd2VkX24pKQogICAgICAgIGlmIG4gbm90IGluIHNlbGYuX2FsbG93ZWRfbjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIm49e259IGlzIG5vdCBpbiBhbGxvd2VkX249e3NlbGYuX2FsbG93ZWRfbn0iKQogICAgICAgIHJldHVybiBuCgogICAgZGVmIF9zZWxlY3RfcG9vbChzZWxmLCBzcGxpdDogc3RyKSAtPiBsaXN0W1BhdGhdOgogICAgICAgIGlmIHNwbGl0ID09ICJoZWxkb3V0IjoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2hlbGRvdXRfZmlsZXMKICAgICAgICBpZiBzcGxpdCA9PSAidHJhaW4iOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fdHJhaW5fZmlsZXMKICAgICAgICByZXR1cm4gc2VsZi5fdHJhaW5fZmlsZXMgKyBzZWxmLl9oZWxkb3V0X2ZpbGVzCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIF9wYWRfYW5kX3N1bSh3YXZlZm9ybXM6IGxpc3RbbnAubmRhcnJheV0pIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgICAgICIiIlplcm8tcGFkIHRvIHRoZSBsb25nZXN0IHN0ZW0sIHN0YWNrIHRvIFtOLCBUXSwgc3VtIHRvIFtUXS4iIiIKICAgICAgICBtYXhfbGVuID0gbWF4KHcuc2hhcGVbMF0gZm9yIHcgaW4gd2F2ZWZvcm1zKQogICAgICAgIHBhZGRlZCA9IFtucC5wYWQodywgKDAsIG1heF9sZW4gLSB3LnNoYXBlWzBdKSkuYXN0eXBlKG5wLmZsb2F0MzIpIGZvciB3IGluIHdhdmVmb3Jtc10KICAgICAgICByZWZzID0gbnAuc3RhY2socGFkZGVkLCBheGlzPTApCiAgICAgICAgcmV0dXJuIHJlZnMsIHJlZnMuc3VtKGF4aXM9MCkK'))
os.makedirs(f'{PROJ}/data', exist_ok=True)
open(f'{PROJ}/data/degradations.py', 'wb').write(base64.b64decode('IiIiCkRlZ3JhZGF0aW9uIGFwcGxpY2F0aW9uIHdpdGggd2V0LXJlZmVyZW5jZSBwb2xpY3kgKERldiBBLCBQMC1BNyBhbmQgUDEtQTEvQTIpLgoKQXBwbGllcyByZXZlcmJlcmF0aW9uLCBub2lzZSwgYW5kIGNvZGVjIGRhbWFnZSB0byBhIGNsZWFuIDgga0h6IG1peHR1cmUgYW5kCmV4dGVuZHMgaXRzIE1peHR1cmVSZWNpcGUgd2l0aCB0aGUgZ3JvdW5kLXRydXRoIGxhYmVscy4gVGhpcyBpcyB0aGUgbW9kdWxlIHRoYXQKdHVybnMgYSBjbGVhbiBDYWxtU2VwTWl4dHVyZSBpbnRvIGEgdHJhaW5pbmcgb3IgZXZhbHVhdGlvbiBleGFtcGxlIHVuZGVyIGEKbmFtZWQgY29uZGl0aW9uLgoKVGhlIHJlZmVyZW5jZSBwb2xpY3kgaXMgdGhlIHN1YnRsZSBwYXJ0LiBCTFVFUFJJTlQgNy42OiBmb3IgcmV2ZXJiZXJhbnQgZGF0YQp0aGUgdGFyZ2V0IGlzIHRoZSAqKndldCBzb3VyY2UqKiAodGhlIHNwZWFrZXIgY29udm9sdmVkIHdpdGggdGhlIFJJUiwgdHJ1bmNhdGVkCmF0IG5fcGVhayArIDUxMiBzYW1wbGVzKSwgbm90IHRoZSBkcnkgc291cmNlLiBUaGUgc3lzdGVtIGlzIGFza2VkIHRvIHNlcGFyYXRlCnNwZWFrZXJzLCBub3QgdG8gZGVyZXZlcmJlcmF0ZSB0aGVtLiBTY29yaW5nIGFnYWluc3QgZHJ5IHNvdXJjZXMgd291bGQgY29uZmxhdGUKdHdvIHRhc2tzIGFuZCBtYWtlIHJldmVyYmVyYW50IFNJLVNEUmkgdW5pbnRlcnByZXRhYmxlOiBhIHBlcmZlY3Qgc2VwYXJhdG9yCnRoYXQgbGVhdmVzIHJldmVyYiBpbnRhY3Qgd291bGQgc2NvcmUgYmFkbHksIHdoaWNoIGlzIHRoZSB3cm9uZyBpbmNlbnRpdmUuCgpUaGUgdHJ1bmNhdGlvbiBvZmZzZXQgKDUxMiBzYW1wbGVzIGF0IDgga0h6ID0gNjQgbXMpIGtlZXBzIHRoZSBkaXJlY3QgcGF0aCBhbmQKdGhlIGVhcmx5IHJlZmxlY3Rpb25zIHRoYXQgYXJyaXZlIHdpdGggaXQsIGFuZCBkaXNjYXJkcyB0aGUgbGF0ZSB0YWlsLiBFYXJseQpyZWZsZWN0aW9ucyBhcmUgcGVyY2VwdHVhbGx5IGZ1c2VkIHdpdGggdGhlIGRpcmVjdCBzb3VuZCBhbmQgY2FycnkgdGhlIHNwZWFrZXIncwp0aW1icmU7IHRoZSBsYXRlIHRhaWwgaXMgd2hhdCBhIGRlcmV2ZXJiZXJhdG9yIHdvdWxkIHJlbW92ZS4gS2VlcGluZyB0aGUgZWFybHkKcGFydCBpbiB0aGUgdGFyZ2V0IGlzIHdoYXQgbWFrZXMgInNlcGFyYXRlIGJ1dCBkbyBub3QgZGVyZXZlcmJlcmF0ZSIgcHJlY2lzZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCByZXBsYWNlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2NpcHkgaW1wb3J0IHNpZ25hbAoKZnJvbSBkYXRhLmNhbG1zZXBfbWl4ZXIgaW1wb3J0IENBTE1TRVBfU0FNUExFX1JBVEUsIENhbG1TZXBNaXh0dXJlLCBNaXh0dXJlUmVjaXBlCmZyb20gZGF0YS5taXhlcl9zdHViIGltcG9ydCBNaXh0dXJlU2FtcGxlCmZyb20gZGF0YS5yaXJfYmFuayBpbXBvcnQgUmlyQmFuaywgUmlyUmVjb3JkLCBzYW1wbGVfdDYwCgpXRVRfUkVGRVJFTkNFX09GRlNFVF9TQU1QTEVTOiBpbnQgPSA1MTIKIiIiCkJMVUVQUklOVCA3LjY6IHdldCByZWZlcmVuY2VzIGFyZSB0cnVuY2F0ZWQgYXQgbl9wZWFrICsgNTEyLiBBdCA4IGtIeiB0aGlzIGlzCjY0IG1zIG9mIGVhcmx5IHJlZmxlY3Rpb25zIHJldGFpbmVkIHBhc3QgdGhlIGRpcmVjdCBwYXRoLiBNYXRjaGVzIHRoZSBzb3VyY2UKcGFwZXIncyBzaW5nbGUtY2hhbm5lbCBuX29mZnNldC4KIiIiCgpTTlJfTUlOX0RCOiBmbG9hdCA9IC02LjAKU05SX01BWF9EQjogZmxvYXQgPSAxMC4wCiIiIkJMVUVQUklOVCA1LjM6IGFkYXB0ZXJfbm9pc2UgdHJhaW5zIG9uIFNOUiB1bmlmb3JtIC02IHRvICsxMCBkQi4iIiIKClNFVkVSRV9TTlJfREI6IGZsb2F0ID0gLTQuMAoiIiIKQkxVRVBSSU5UIDcuNSBob2xkb3V0IDM6IFNOUiBiZWxvdyAtNCBkQiBpcyBhIHNldmVyaXR5IGhvbGRvdXQsIGtlcHQgdG8gMTAlIG9mCm5vaXNlIHRyYWluaW5nIHNhbXBsZXMgYW5kIHByb2JlZCBpbiBldmFsdWF0aW9uLgoiIiIKClNFVkVSRV9GUkFDVElPTjogZmxvYXQgPSAwLjEwCiIiIkZyYWN0aW9uIG9mIHRyYWluaW5nIGRyYXdzIGFsbG93ZWQgaW50byB0aGUgc2V2ZXJlIChTTlIgPCAtNCBkQikgYmFuZC4iIiIKCkVQUzogZmxvYXQgPSAxZS0xMAoiIiJFbmVyZ3kgZ3VhcmQuIE1hdGNoZXMgZXZhbC9tZXRyaWNzLnB5IEVQUyBzY2FsZS4iIiIKCgpkZWYgc2FtcGxlX3NucigKICAgIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvciwKICAgIGFsbG93X3NldmVyZTogYm9vbCA9IFRydWUsCiAgICBzZXZlcmVfZnJhY3Rpb246IGZsb2F0ID0gU0VWRVJFX0ZSQUNUSU9OLAopIC0+IGZsb2F0OgogICAgIiIiCiAgICBEcmF3IGEgdHJhaW5pbmcgU05SLCBob25vdXJpbmcgdGhlIHNldmVyaXR5IGhvbGRvdXQuCgogICAgTWlycm9ycyBkYXRhLnJpcl9iYW5rLnNhbXBsZV90NjAgZm9yIHRoZSBub2lzZSBheGlzLiBCTFVFUFJJTlQgNy41CiAgICBob2xkb3V0IDMga2VlcHMgU05SIGJlbG93IC00IGRCIHJhcmUgaW4gdHJhaW5pbmcgc28gZXZhbHVhdGlvbiBhdCBsb3cgU05SCiAgICBtZWFzdXJlcyBleHRyYXBvbGF0aW9uLgoKICAgIEFyZ3M6CiAgICAgICAgcm5nOiBTZWVkZWQgZ2VuZXJhdG9yLgogICAgICAgIGFsbG93X3NldmVyZTogV2hlbiBGYWxzZSwgbmV2ZXIgZHJhd3MgYmVsb3cgU0VWRVJFX1NOUl9EQi4KICAgICAgICBzZXZlcmVfZnJhY3Rpb246IFByb2JhYmlsaXR5IG9mIGRyYXdpbmcgZnJvbSB0aGUgc2V2ZXJlIGJhbmQuCgogICAgUmV0dXJuczoKICAgICAgICBTTlIgaW4gZEIsIHdpdGhpbiBbU05SX01JTl9EQiwgU05SX01BWF9EQl0uCiAgICAiIiIKICAgIGlmIG5vdCBhbGxvd19zZXZlcmU6CiAgICAgICAgcmV0dXJuIGZsb2F0KHJuZy51bmlmb3JtKFNFVkVSRV9TTlJfREIsIFNOUl9NQVhfREIpKQogICAgaWYgcm5nLnJhbmRvbSgpIDwgc2V2ZXJlX2ZyYWN0aW9uOgogICAgICAgIHJldHVybiBmbG9hdChybmcudW5pZm9ybShTTlJfTUlOX0RCLCBTRVZFUkVfU05SX0RCKSkKICAgIHJldHVybiBmbG9hdChybmcudW5pZm9ybShTRVZFUkVfU05SX0RCLCBTTlJfTUFYX0RCKSkKCgpkZWYgbWFrZV93ZXRfcmVmZXJlbmNlKAogICAgZHJ5OiBucC5uZGFycmF5LAogICAgcmlyOiBucC5uZGFycmF5LAogICAgbl9wZWFrOiBpbnQsCiAgICBvZmZzZXQ6IGludCA9IFdFVF9SRUZFUkVOQ0VfT0ZGU0VUX1NBTVBMRVMsCiAgICB0YXJnZXRfbGVuZ3RoOiBpbnQgfCBOb25lID0gTm9uZSwKKSAtPiBucC5uZGFycmF5OgogICAgIiIiCiAgICBDb252b2x2ZSBhIGRyeSBzb3VyY2Ugd2l0aCBhIHRydW5jYXRlZCBSSVIgdG8gbWFrZSB0aGUgd2V0IHJlZmVyZW5jZS4KCiAgICBUaGUgUklSIGlzIGN1dCBhdCBuX3BlYWsgKyBvZmZzZXQgYmVmb3JlIGNvbnZvbHV0aW9uLCBzbyB0aGUgcmVmZXJlbmNlCiAgICBjb250YWlucyB0aGUgZGlyZWN0IHBhdGggYW5kIGVhcmx5IHJlZmxlY3Rpb25zIGJ1dCBub3QgdGhlIGxhdGUgdGFpbC4gU2VlCiAgICB0aGUgbW9kdWxlIGRvY3N0cmluZyBmb3Igd2h5IHRoaXMgaXMgdGhlIGNvcnJlY3QgdGFyZ2V0LgoKICAgIEFyZ3M6CiAgICAgICAgZHJ5OiBDbGVhbiBzb3VyY2Ugd2F2ZWZvcm0gW1RdLgogICAgICAgIHJpcjogRnVsbCByb29tIGltcHVsc2UgcmVzcG9uc2UgW1JdLgogICAgICAgIG5fcGVhazogSW5kZXggb2YgdGhlIGRpcmVjdC1wYXRoIHBlYWsgaW4gcmlyIChmcm9tIFJpclJlY29yZC5uX3BlYWspLgogICAgICAgIG9mZnNldDogU2FtcGxlcyBrZXB0IHBhc3QgdGhlIHBlYWsuIERlZmF1bHRzIHRvIDUxMiAoNjQgbXMgYXQgOCBrSHopLgogICAgICAgIHRhcmdldF9sZW5ndGg6IENyb3Agb3IgcGFkIHRoZSByZXN1bHQgdG8gZXhhY3RseSB0aGlzIG1hbnkgc2FtcGxlcy4KICAgICAgICAgICAgRGVmYXVsdHMgdG8gbGVuKGRyeSksIHdoaWNoIGtlZXBzIHRoZSByZWZlcmVuY2UgdGltZS1hbGlnbmVkIHdpdGgKICAgICAgICAgICAgdGhlIGRyeSBzb3VyY2Ugc28gU0ktU0RSIGlzIG1lYW5pbmdmdWwuCgogICAgUmV0dXJuczoKICAgICAgICBXZXQgcmVmZXJlbmNlIFt0YXJnZXRfbGVuZ3RoXSBmbG9hdDMyLgogICAgIiIiCiAgICBkID0gbnAuYXNhcnJheShkcnksIGR0eXBlPW5wLmZsb2F0MzIpLnNxdWVlemUoKQogICAgaCA9IG5wLmFzYXJyYXkocmlyLCBkdHlwZT1ucC5mbG9hdDMyKS5zcXVlZXplKCkKICAgIGlmIGQubmRpbSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJkcnkgbXVzdCBiZSAxLUQsIGdvdCBzaGFwZSB7ZC5zaGFwZX0iKQogICAgaWYgaC5uZGltICE9IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInJpciBtdXN0IGJlIDEtRCwgZ290IHNoYXBlIHtoLnNoYXBlfSIpCgogICAgY3V0ID0gbWluKG5fcGVhayArIG9mZnNldCwgaC5zaGFwZVswXSkKICAgIGlmIGN1dCA8PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ0cnVuY2F0aW9uIGluZGV4IHtjdXR9IGlzIG5vdCBwb3NpdGl2ZSAobl9wZWFrPXtuX3BlYWt9KSIpCiAgICBoX2Vhcmx5ID0gaFs6Y3V0XQoKICAgIHdldCA9IHNpZ25hbC5mZnRjb252b2x2ZShkLCBoX2Vhcmx5LCBtb2RlPSJmdWxsIikKICAgIGxlbmd0aCA9IGludCh0YXJnZXRfbGVuZ3RoKSBpZiB0YXJnZXRfbGVuZ3RoIGlzIG5vdCBOb25lIGVsc2UgZC5zaGFwZVswXQogICAgcmV0dXJuIF9maXRfbGVuZ3RoKHdldCwgbGVuZ3RoKS5hc3R5cGUobnAuZmxvYXQzMikKCgpkZWYgbWFrZV93ZXRfbWl4dHVyZSgKICAgIGRyeTogbnAubmRhcnJheSwKICAgIHJpcjogbnAubmRhcnJheSwKICAgIHRhcmdldF9sZW5ndGg6IGludCB8IE5vbmUgPSBOb25lLAopIC0+IG5wLm5kYXJyYXk6CiAgICAiIiIKICAgIENvbnZvbHZlIGEgZHJ5IHNvdXJjZSB3aXRoIHRoZSAqKmZ1bGwqKiBSSVIgdG8gbWFrZSB0aGUgb2JzZXJ2ZWQgc2lnbmFsLgoKICAgIFRoZSBtaXh0dXJlIHRoZSBzeXN0ZW0gaGVhcnMgY2FycmllcyB0aGUgY29tcGxldGUgcmV2ZXJiZXJhdGlvbiBpbmNsdWRpbmcKICAgIHRoZSBsYXRlIHRhaWwuIE9ubHkgdGhlICpyZWZlcmVuY2UqIGlzIHRydW5jYXRlZC4gVXNpbmcgdGhlIHRydW5jYXRlZCBSSVIKICAgIGZvciBib3RoIHdvdWxkIG1lYW4gdGhlIHN5c3RlbSBuZXZlciBzZWVzIHRoZSB0YWlsIGl0IG11c3QgYmUgcm9idXN0IHRvLgoKICAgIEFyZ3M6CiAgICAgICAgZHJ5OiBDbGVhbiBzb3VyY2Ugd2F2ZWZvcm0gW1RdLgogICAgICAgIHJpcjogRnVsbCByb29tIGltcHVsc2UgcmVzcG9uc2UgW1JdLgogICAgICAgIHRhcmdldF9sZW5ndGg6IE91dHB1dCBsZW5ndGguIERlZmF1bHRzIHRvIGxlbihkcnkpLgoKICAgIFJldHVybnM6CiAgICAgICAgUmV2ZXJiZXJhbnQgb2JzZXJ2YXRpb24gW3RhcmdldF9sZW5ndGhdIGZsb2F0MzIuCiAgICAiIiIKICAgIGQgPSBucC5hc2FycmF5KGRyeSwgZHR5cGU9bnAuZmxvYXQzMikuc3F1ZWV6ZSgpCiAgICBoID0gbnAuYXNhcnJheShyaXIsIGR0eXBlPW5wLmZsb2F0MzIpLnNxdWVlemUoKQogICAgd2V0ID0gc2lnbmFsLmZmdGNvbnZvbHZlKGQsIGgsIG1vZGU9ImZ1bGwiKQogICAgbGVuZ3RoID0gaW50KHRhcmdldF9sZW5ndGgpIGlmIHRhcmdldF9sZW5ndGggaXMgbm90IE5vbmUgZWxzZSBkLnNoYXBlWzBdCiAgICByZXR1cm4gX2ZpdF9sZW5ndGgod2V0LCBsZW5ndGgpLmFzdHlwZShucC5mbG9hdDMyKQoKCmRlZiBhcHBseV9yZXZlcmIoCiAgICBtaXh0dXJlOiBDYWxtU2VwTWl4dHVyZSwKICAgIHJpcl9iYW5rOiBSaXJCYW5rLAogICAgcm5nOiBucC5yYW5kb20uR2VuZXJhdG9yLAogICAgdDYwX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUsCiAgICBhbGxvd19zZXZlcmU6IGJvb2wgPSBUcnVlLAogICAgcmVjb3JkOiBSaXJSZWNvcmQgfCBOb25lID0gTm9uZSwKKSAtPiBDYWxtU2VwTWl4dHVyZToKICAgICIiIgogICAgUmV2ZXJiZXJhdGUgZXZlcnkgc3RlbSB3aXRoIG9uZSBzaGFyZWQgcm9vbSwgYW5kIHJlYnVpbGQgdGhlIG1peHR1cmUuCgogICAgQWxsIHNwZWFrZXJzIHNoYXJlIHRoZSBzYW1lIFJJUiBiZWNhdXNlIHRoZXkgYXJlIGluIHRoZSBzYW1lIHJvb20uIERyYXdpbmcKICAgIGEgc2VwYXJhdGUgUklSIHBlciBzcGVha2VyIHdvdWxkIG1vZGVsIGFuIGFjb3VzdGljYWxseSBpbXBvc3NpYmxlIHNjZW5lIGFuZAogICAgd291bGQgZ2l2ZSB0aGUgbW9kZWwgYSBzcHVyaW91cyBwZXItc3BlYWtlciBjdWUgdG8gc2VwYXJhdGUgb24uCgogICAgVGhlIHJldHVybmVkIG1peHR1cmUncyByZWZlcmVuY2VzIGFyZSAqKndldCoqICh0cnVuY2F0ZWQgUklSKSBhbmQgaXRzCiAgICBvYnNlcnZhdGlvbiBpcyAqKmZ1bGx5IHJldmVyYmVyYW50KiogKGZ1bGwgUklSKS4gVGhlIHJlY2lwZSByZWNvcmRzIHRoZQogICAgYWNoaWV2ZWQgVDYwIGFuZCB0aGUgUklSIHBhdGguCgogICAgQXJnczoKICAgICAgICBtaXh0dXJlOiBBIGNsZWFuIENhbG1TZXBNaXh0dXJlLgogICAgICAgIHJpcl9iYW5rOiBMb2FkZWQgc2ltdWxhdGVkIFJJUiBiYW5rLgogICAgICAgIHJuZzogU2VlZGVkIGdlbmVyYXRvci4KICAgICAgICB0NjBfczogVGFyZ2V0IFQ2MC4gRHJhd24gdmlhIHNhbXBsZV90NjAgd2hlbiBOb25lLgogICAgICAgIGFsbG93X3NldmVyZTogUGFzc2VkIHRvIHNhbXBsZV90NjAgZm9yIHRoZSBzZXZlcml0eSBob2xkb3V0LgogICAgICAgIHJlY29yZDogVXNlIHRoaXMgZXhhY3QgUklSIGluc3RlYWQgb2YgZHJhd2luZyBvbmUuIFNldCBieSB0aGUgZml4ZWQKICAgICAgICAgICAgZXZhbHVhdGlvbiBnZW5lcmF0b3Igc28gYW4gZXZhbCBjZWxsIHBpbnMgaXRzIHJvb21zLgoKICAgIFJldHVybnM6CiAgICAgICAgQSBuZXcgQ2FsbVNlcE1peHR1cmUuIFRoZSBpbnB1dCBpcyBub3QgbW9kaWZpZWQuCiAgICAiIiIKICAgIGlmIHJlY29yZCBpcyBOb25lOgogICAgICAgIHRhcmdldCA9IHQ2MF9zIGlmIHQ2MF9zIGlzIG5vdCBOb25lIGVsc2Ugc2FtcGxlX3Q2MChybmcsIGFsbG93X3NldmVyZT1hbGxvd19zZXZlcmUpCiAgICAgICAgcmVjb3JkID0gcmlyX2Jhbmsuc2FtcGxlKHRhcmdldCkKICAgIHJpciA9IHJpcl9iYW5rLmxvYWQocmVjb3JkKQoKICAgIHJlZnMgPSBtaXh0dXJlLnJlZmVyZW5jZXMKICAgIGxlbmd0aCA9IHJlZnMuc2hhcGVbMV0KCiAgICB3ZXRfcmVmcyA9IG5wLnN0YWNrKAogICAgICAgIFttYWtlX3dldF9yZWZlcmVuY2UocmVmc1tpXSwgcmlyLCByZWNvcmQubl9wZWFrLCB0YXJnZXRfbGVuZ3RoPWxlbmd0aCkgZm9yIGkgaW4gcmFuZ2UocmVmcy5zaGFwZVswXSldLAogICAgICAgIGF4aXM9MCwKICAgICkKICAgIHdldF9vYnMgPSBucC5zdGFjaygKICAgICAgICBbbWFrZV93ZXRfbWl4dHVyZShyZWZzW2ldLCByaXIsIHRhcmdldF9sZW5ndGg9bGVuZ3RoKSBmb3IgaSBpbiByYW5nZShyZWZzLnNoYXBlWzBdKV0sCiAgICAgICAgYXhpcz0wLAogICAgKQogICAgb2JzZXJ2ZWQgPSB3ZXRfb2JzLnN1bShheGlzPTApLmFzdHlwZShucC5mbG9hdDMyKQoKICAgIHJlY2lwZSA9IHJlcGxhY2UoCiAgICAgICAgbWl4dHVyZS5yZWNpcGUsCiAgICAgICAgdDYwX3M9cmVjb3JkLnQ2MF9hY2hpZXZlZF9zLAogICAgICAgIHJpcl9maWxlPXJlY29yZC5wYXRoLAogICAgKQogICAgc2FtcGxlID0gTWl4dHVyZVNhbXBsZSgKICAgICAgICBtaXh0dXJlPW9ic2VydmVkLAogICAgICAgIHJlZmVyZW5jZXM9d2V0X3JlZnMsCiAgICAgICAgc2FtcGxlX3JhdGU9bWl4dHVyZS5zYW1wbGUuc2FtcGxlX3JhdGUsCiAgICAgICAgdXR0ZXJhbmNlX2lkPW1peHR1cmUuc2FtcGxlLnV0dGVyYW5jZV9pZCwKICAgICkKICAgIHJldHVybiBDYWxtU2VwTWl4dHVyZShzYW1wbGU9c2FtcGxlLCByZWNpcGU9cmVjaXBlKQoKCmRlZiBhcHBseV9ub2lzZSgKICAgIG1peHR1cmU6IENhbG1TZXBNaXh0dXJlLAogICAgbm9pc2U6IG5wLm5kYXJyYXksCiAgICBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IsCiAgICBzbnJfZGI6IGZsb2F0IHwgTm9uZSA9IE5vbmUsCiAgICBhbGxvd19zZXZlcmU6IGJvb2wgPSBUcnVlLAogICAgbm9pc2VfZmlsZTogc3RyIHwgTm9uZSA9IE5vbmUsCikgLT4gQ2FsbVNlcE1peHR1cmU6CiAgICAiIiIKICAgIEFkZCBiYWNrZ3JvdW5kIG5vaXNlIGF0IGEgZHJhd24gU05SLCBsZWF2aW5nIHRoZSByZWZlcmVuY2VzIHVudG91Y2hlZC4KCiAgICBUaGUgcmVmZXJlbmNlcyBkbyBub3QgY2hhbmdlOiBub2lzZSBpcyBub3QgYSBzcGVha2VyLCBzbyByZW1vdmluZyBpdCBpcwogICAgcGFydCBvZiB0aGUgdGFzaywgbm90IHBhcnQgb2YgdGhlIHRhcmdldC4gVGhlIFNOUiBpcyBjb21wdXRlZCBhZ2FpbnN0IHRoZQogICAgc3BlZWNoIG1peHR1cmUncyBlbmVyZ3kgc28gdGhlIGxhYmVsIG1lYW5zIHdoYXQgaXQgc2F5cy4KCiAgICBBcmdzOgogICAgICAgIG1peHR1cmU6IEEgQ2FsbVNlcE1peHR1cmUsIGNsZWFuIG9yIGFscmVhZHkgcmV2ZXJiZXJhdGVkLgogICAgICAgIG5vaXNlOiBOb2lzZSB3YXZlZm9ybSBbVF9uXSBhdCB0aGUgbWl4dHVyZSdzIHJhdGUuIExvb3BlZCBvciBjcm9wcGVkIHRvCiAgICAgICAgICAgIHRoZSBtaXh0dXJlJ3MgbGVuZ3RoLgogICAgICAgIHJuZzogU2VlZGVkIGdlbmVyYXRvciwgdXNlZCBmb3IgdGhlIFNOUiBkcmF3IGFuZCB0aGUgbm9pc2Ugb2Zmc2V0LgogICAgICAgIHNucl9kYjogVGFyZ2V0IFNOUi4gRHJhd24gdmlhIHNhbXBsZV9zbnIgd2hlbiBOb25lLgogICAgICAgIGFsbG93X3NldmVyZTogUGFzc2VkIHRvIHNhbXBsZV9zbnIgZm9yIHRoZSBzZXZlcml0eSBob2xkb3V0LgogICAgICAgIG5vaXNlX2ZpbGU6IFBhdGggcmVjb3JkZWQgaW50byB0aGUgcmVjaXBlIGZvciB0cmFjZWFiaWxpdHkuCgogICAgUmV0dXJuczoKICAgICAgICBBIG5ldyBDYWxtU2VwTWl4dHVyZSB3aXRoIG5vaXNlIGFkZGVkIHRvIHRoZSBvYnNlcnZhdGlvbi4KICAgICIiIgogICAgdGFyZ2V0X3NuciA9IHNucl9kYiBpZiBzbnJfZGIgaXMgbm90IE5vbmUgZWxzZSBzYW1wbGVfc25yKHJuZywgYWxsb3dfc2V2ZXJlPWFsbG93X3NldmVyZSkKCiAgICBzcGVlY2ggPSBtaXh0dXJlLm1peHR1cmUKICAgIGxlbmd0aCA9IHNwZWVjaC5zaGFwZVswXQogICAgbm9pc2VfZml0ID0gX2xvb3Bfb3JfY3JvcChucC5hc2FycmF5KG5vaXNlLCBkdHlwZT1ucC5mbG9hdDMyKS5zcXVlZXplKCksIGxlbmd0aCwgcm5nKQoKICAgIHNwZWVjaF9wb3dlciA9IGZsb2F0KG5wLm1lYW4oc3BlZWNoKioyKSkKICAgIG5vaXNlX3Bvd2VyID0gZmxvYXQobnAubWVhbihub2lzZV9maXQqKjIpKQogICAgaWYgbm9pc2VfcG93ZXIgPCBFUFM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibm9pc2Ugc2VnbWVudCBpcyBzaWxlbnQ7IGNhbm5vdCBzY2FsZSBpdCB0byBhIHRhcmdldCBTTlIiKQogICAgaWYgc3BlZWNoX3Bvd2VyIDwgRVBTOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNwZWVjaCBtaXh0dXJlIGlzIHNpbGVudDsgU05SIGlzIHVuZGVmaW5lZCIpCgogICAgIyBzY2FsZSBzbyB0aGF0IDEwKmxvZzEwKHNwZWVjaF9wb3dlciAvIChub2lzZV9wb3dlciAqIHNjYWxlXjIpKSA9PSB0YXJnZXRfc25yCiAgICBzY2FsZSA9IGZsb2F0KG5wLnNxcnQoc3BlZWNoX3Bvd2VyIC8gKG5vaXNlX3Bvd2VyICogMTAuMCAqKiAodGFyZ2V0X3NuciAvIDEwLjApKSkpCiAgICBub2lzeSA9IChzcGVlY2ggKyBub2lzZV9maXQgKiBzY2FsZSkuYXN0eXBlKG5wLmZsb2F0MzIpCgogICAgcmVjaXBlID0gcmVwbGFjZShtaXh0dXJlLnJlY2lwZSwgc25yX2RiPXRhcmdldF9zbnIsIG5vaXNlX2ZpbGU9bm9pc2VfZmlsZSkKICAgIHNhbXBsZSA9IE1peHR1cmVTYW1wbGUoCiAgICAgICAgbWl4dHVyZT1ub2lzeSwKICAgICAgICByZWZlcmVuY2VzPW1peHR1cmUucmVmZXJlbmNlcywKICAgICAgICBzYW1wbGVfcmF0ZT1taXh0dXJlLnNhbXBsZS5zYW1wbGVfcmF0ZSwKICAgICAgICB1dHRlcmFuY2VfaWQ9bWl4dHVyZS5zYW1wbGUudXR0ZXJhbmNlX2lkLAogICAgKQogICAgcmV0dXJuIENhbG1TZXBNaXh0dXJlKHNhbXBsZT1zYW1wbGUsIHJlY2lwZT1yZWNpcGUpCgoKZGVmIGFwcGx5X2NvZGVjKAogICAgbWl4dHVyZTogQ2FsbVNlcE1peHR1cmUsCiAgICBjb2RlY19uYW1lOiBzdHIsCiAgICBiaXRyYXRlX2JwczogaW50LAogICAgdG1wX2Rpcjogc3RyIHwgUGF0aCB8IE5vbmUgPSBOb25lLAopIC0+IENhbG1TZXBNaXh0dXJlOgogICAgIiIiCiAgICBSb3VuZC10cmlwIHRoZSBvYnNlcnZhdGlvbiB0aHJvdWdoIGEgbG9zc3kgY29kZWMsIGxlYXZpbmcgcmVmZXJlbmNlcyBjbGVhbi4KCiAgICBMaWtlIG5vaXNlLCBjb2RlYyBkYW1hZ2UgaXMgc29tZXRoaW5nIHRoZSBzeXN0ZW0gbXVzdCB1bmRvLCBzbyBpdCBpcyBhcHBsaWVkCiAgICB0byB0aGUgb2JzZXJ2YXRpb24gb25seS4gZGF0YS9jb2RlY19hdWdtZW50YXRpb24ucHkgb3ducyB0aGUgZmZtcGVnIGNhbGxzOwogICAgdGhpcyB3cmFwcGVyIGFkYXB0cyB0aGVtIHRvIHRoZSBDYWxtU2VwTWl4dHVyZSB0eXBlIGFuZCByZWNvcmRzIHRoZSBsYWJlbHMuCgogICAgQXJnczoKICAgICAgICBtaXh0dXJlOiBBIENhbG1TZXBNaXh0dXJlIGF0IDgga0h6LgogICAgICAgIGNvZGVjX25hbWU6IE9uZSBvZiAib3B1cyIsICJhYWMiLCAiYW1yLW5iIiwgImFtci13YiIuCiAgICAgICAgYml0cmF0ZV9icHM6IFRhcmdldCBiaXRyYXRlIGluIGJpdHMgcGVyIHNlY29uZC4KICAgICAgICB0bXBfZGlyOiBTY3JhdGNoIGRpcmVjdG9yeSBmb3IgdGhlIGVuY29kZS9kZWNvZGUgcm91bmQgdHJpcC4KCiAgICBSZXR1cm5zOgogICAgICAgIEEgbmV3IENhbG1TZXBNaXh0dXJlIHdpdGggYSBjb2RlYy1kYW1hZ2VkIG9ic2VydmF0aW9uLgogICAgIiIiCiAgICBmcm9tIGRhdGEuY29kZWNfYXVnbWVudGF0aW9uIGltcG9ydCBhcHBseV9jb2RlY19yb3VuZHRyaXAKCiAgICBkYW1hZ2VkID0gYXBwbHlfY29kZWNfcm91bmR0cmlwKAogICAgICAgIGF1ZGlvPW1peHR1cmUubWl4dHVyZSwKICAgICAgICBzYW1wbGVfcmF0ZT1taXh0dXJlLnNhbXBsZS5zYW1wbGVfcmF0ZSwKICAgICAgICBjb2RlYz1jb2RlY19uYW1lLAogICAgICAgIGJpdHJhdGVfYnBzPWJpdHJhdGVfYnBzLAogICAgICAgIHRtcF9kaXI9dG1wX2RpciwKICAgICkKICAgIGRhbWFnZWQgPSBfZml0X2xlbmd0aChkYW1hZ2VkLCBtaXh0dXJlLm1peHR1cmUuc2hhcGVbMF0pLmFzdHlwZShucC5mbG9hdDMyKQoKICAgIHJlY2lwZSA9IHJlcGxhY2UobWl4dHVyZS5yZWNpcGUsIGNvZGVjX25hbWU9Y29kZWNfbmFtZSwgY29kZWNfYml0cmF0ZV9icHM9aW50KGJpdHJhdGVfYnBzKSkKICAgIHNhbXBsZSA9IE1peHR1cmVTYW1wbGUoCiAgICAgICAgbWl4dHVyZT1kYW1hZ2VkLAogICAgICAgIHJlZmVyZW5jZXM9bWl4dHVyZS5yZWZlcmVuY2VzLAogICAgICAgIHNhbXBsZV9yYXRlPW1peHR1cmUuc2FtcGxlLnNhbXBsZV9yYXRlLAogICAgICAgIHV0dGVyYW5jZV9pZD1taXh0dXJlLnNhbXBsZS51dHRlcmFuY2VfaWQsCiAgICApCiAgICByZXR1cm4gQ2FsbVNlcE1peHR1cmUoc2FtcGxlPXNhbXBsZSwgcmVjaXBlPXJlY2lwZSkKCgpkZWYgX2ZpdF9sZW5ndGgoeDogbnAubmRhcnJheSwgbGVuZ3RoOiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJDcm9wIG9yIHplcm8tcGFkIGEgMS1EIHNpZ25hbCB0byBleGFjdGx5IGBsZW5ndGhgIHNhbXBsZXMuIiIiCiAgICB0ID0geC5zaGFwZVswXQogICAgaWYgdCA9PSBsZW5ndGg6CiAgICAgICAgcmV0dXJuIHgKICAgIGlmIHQgPiBsZW5ndGg6CiAgICAgICAgcmV0dXJuIHhbOmxlbmd0aF0KICAgIHJldHVybiBucC5wYWQoeCwgKDAsIGxlbmd0aCAtIHQpKQoKCmRlZiBfbG9vcF9vcl9jcm9wKG5vaXNlOiBucC5uZGFycmF5LCBsZW5ndGg6IGludCwgcm5nOiBucC5yYW5kb20uR2VuZXJhdG9yKSAtPiBucC5uZGFycmF5OgogICAgIiIiCiAgICBGaXQgYSBub2lzZSBjbGlwIHRvIGBsZW5ndGhgIGJ5IGxvb3BpbmcgaXQgb3IgY3JvcHBpbmcgYSByYW5kb20gd2luZG93LgoKICAgIEEgcmFuZG9tIG9mZnNldCBpcyB1c2VkIHJhdGhlciB0aGFuIGFsd2F5cyBzdGFydGluZyBhdCBzYW1wbGUgMCwgc28gYSBsb25nCiAgICBub2lzZSBmaWxlIGNvbnRyaWJ1dGVzIG1hbnkgZGlzdGluY3Qgc2VnbWVudHMgYWNyb3NzIGFuIGVwb2NoIGluc3RlYWQgb2YKICAgIHRoZSBzYW1lIG9wZW5pbmcgZXZlcnkgdGltZS4KICAgICIiIgogICAgbiA9IG5vaXNlLnNoYXBlWzBdCiAgICBpZiBuID09IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibm9pc2UgY2xpcCBpcyBlbXB0eSIpCiAgICBpZiBuIDwgbGVuZ3RoOgogICAgICAgIHJlcHMgPSBpbnQobnAuY2VpbChsZW5ndGggLyBuKSkKICAgICAgICByZXR1cm4gbnAudGlsZShub2lzZSwgcmVwcylbOmxlbmd0aF0KICAgIHN0YXJ0ID0gaW50KHJuZy5pbnRlZ2VycygwLCBuIC0gbGVuZ3RoICsgMSkpCiAgICByZXR1cm4gbm9pc2Vbc3RhcnQgOiBzdGFydCArIGxlbmd0aF0KCgpkZWYgZGVzY3JpYmVfY29uZGl0aW9uKHJlY2lwZTogTWl4dHVyZVJlY2lwZSkgLT4gc3RyOgogICAgIiIiCiAgICBOYW1lIHRoZSBjb25kaXRpb24gY2VsbCBhIHJlY2lwZSBiZWxvbmdzIHRvLgoKICAgIFVzZWQgdG8ga2V5IHRoZSBldmFsdWF0aW9uIG1hdHJpeCAoQkxVRVBSSU5UIDkuNCkgYW5kIHRvIGNoZWNrIHRoZQogICAgY29uZGl0aW9uLWNvbWJpbmF0aW9uIGhvbGRvdXQgKDcuNSwgaG9sZG91dCAyKS4KCiAgICBSZXR1cm5zOgogICAgICAgIE9uZSBvZiAiY2xlYW4iLCAicmV2ZXJiIiwgIm5vaXNlIiwgImNvZGVjIiwgInJldmVyYitub2lzZSIsCiAgICAgICAgInJldmVyYitjb2RlYyIsICJub2lzZStjb2RlYyIsICJhbGwtdGhyZWUiLgogICAgIiIiCiAgICBwYXJ0czogbGlzdFtzdHJdID0gW10KICAgIGlmIHJlY2lwZS50NjBfcyBpcyBub3QgTm9uZSBhbmQgcmVjaXBlLnQ2MF9zID4gMC4wOgogICAgICAgIHBhcnRzLmFwcGVuZCgicmV2ZXJiIikKICAgIGlmIHJlY2lwZS5zbnJfZGIgaXMgbm90IE5vbmU6CiAgICAgICAgcGFydHMuYXBwZW5kKCJub2lzZSIpCiAgICBpZiByZWNpcGUuY29kZWNfbmFtZSBpcyBub3QgTm9uZToKICAgICAgICBwYXJ0cy5hcHBlbmQoImNvZGVjIikKCiAgICBpZiBub3QgcGFydHM6CiAgICAgICAgcmV0dXJuICJjbGVhbiIKICAgIGlmIGxlbihwYXJ0cykgPT0gMzoKICAgICAgICByZXR1cm4gImFsbC10aHJlZSIKICAgIHJldHVybiAiKyIuam9pbihwYXJ0cykKCgpIRUxEX09VVF9DT01CSU5BVElPTlM6IGZyb3plbnNldFtzdHJdID0gZnJvemVuc2V0KHsicmV2ZXJiK2NvZGVjIiwgIm5vaXNlK2NvZGVjIn0pCiIiIgpCTFVFUFJJTlQgNy41IGhvbGRvdXQgMjogdGhlc2UgY29tYmluYXRpb25zIG5ldmVyIGFwcGVhciBpbiBnYXRlIG9yIGpvaW50CnRyYWluaW5nIGFuZCBleGlzdCBvbmx5IGluIHRoZSBldmFsdWF0aW9uIG1hdHJpeCwgc28gY29tcG9zaXRpb25hbApnZW5lcmFsaXNhdGlvbiBpcyBtZWFzdXJlZCByYXRoZXIgdGhhbiBhc3N1bWVkLiBhc3NlcnRfbm90X2hlbGRfb3V0IGVuZm9yY2VzIGl0LgoiIiIKCgpkZWYgYXNzZXJ0X25vdF9oZWxkX291dChyZWNpcGU6IE1peHR1cmVSZWNpcGUpIC0+IE5vbmU6CiAgICAiIiIKICAgIFJhaXNlIGlmIGEgcmVjaXBlIGJlbG9uZ3MgdG8gYSBoZWxkLW91dCBjb21iaW5hdGlvbiBjZWxsLgoKICAgIENhbGxlZCBieSB0aGUgZ2F0ZSBhbmQgam9pbnQtcG9saXNoIHRyYWluaW5nIGRhdGEgcGlwZWxpbmVzLiBBIGhlbGQtb3V0CiAgICBjb21iaW5hdGlvbiByZWFjaGluZyB0cmFpbmluZyBzaWxlbnRseSBpbnZhbGlkYXRlcyB0aGUgY29tcG9zaXRpb25hbAogICAgZ2VuZXJhbGlzYXRpb24gY2xhaW0gaW4gQkxVRVBSSU5UIDkuNSBhbmFseXNpcyAzLCBzbyB0aGlzIHJhaXNlcy4KICAgICIiIgogICAgY2VsbCA9IGRlc2NyaWJlX2NvbmRpdGlvbihyZWNpcGUpCiAgICBpZiBjZWxsIGluIEhFTERfT1VUX0NPTUJJTkFUSU9OUzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmInJlY2lwZSBpcyBpbiBoZWxkLW91dCBjb21iaW5hdGlvbiBjZWxsIHtjZWxsIXJ9OyBpdCBtdXN0IG5vdCBlbnRlciBnYXRlIG9yICIKICAgICAgICAgICAgZiJqb2ludCB0cmFpbmluZyAoQkxVRVBSSU5UIDcuNSBob2xkb3V0IDIpLiBIZWxkIG91dDoge3NvcnRlZChIRUxEX09VVF9DT01CSU5BVElPTlMpfSIKICAgICAgICApCg=='))
os.makedirs(f'{PROJ}/data', exist_ok=True)
open(f'{PROJ}/data/rir_bank.py', 'wb').write(base64.b64decode('...'))
os.makedirs(f'{PROJ}/data', exist_ok=True)
open(f'{PROJ}/data/mixer_stub.py', 'wb').write(base64.b64decode('IiIiCk1pbmltYWwgbWl4ZXIgc3R1YiBmb3IgUGhhc2UgMCBiYXNlbGluZSBydW5zLgoKRGV2IEEgd2lsbCByZXBsYWNlIHRoaXMgd2l0aCB0aGUgZnVsbCBkeW5hbWljIG1peGVyIChgZGF0YS9taXhlci5weWApLgpUaGlzIHN0dWIgbG9hZHMgcHJlLW1peGVkIExpYnJpM01peCB0ZXN0IGZpbGVzIGZyb20gZGlzayBhbmQgeWllbGRzCihtaXh0dXJlLCByZWZlcmVuY2Vfc3RlbXMsIHNhbXBsZV9yYXRlKSB0dXBsZXMgZm9yIHRoZSBiYXNlbGluZSBydW5uZXIuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBzb3VuZGZpbGUgYXMgc2YKCgpAZGF0YWNsYXNzCmNsYXNzIE1peHR1cmVTYW1wbGU6CiAgICAiIiJBIHNpbmdsZSBtaXh0dXJlIHdpdGggZ3JvdW5kLXRydXRoIGNsZWFuIHN0ZW1zLiIiIgoKICAgIG1peHR1cmU6IG5wLm5kYXJyYXkKICAgICIiIk1vbm8gbWl4dHVyZSB3YXZlZm9ybSwgc2hhcGUgW1RdLiIiIgoKICAgIHJlZmVyZW5jZXM6IG5wLm5kYXJyYXkKICAgICIiIkNsZWFuIHNvdXJjZSB3YXZlZm9ybXMsIHNoYXBlIFtOLCBUXS4iIiIKCiAgICBzYW1wbGVfcmF0ZTogaW50CiAgICB1dHRlcmFuY2VfaWQ6IHN0cgoKCmRlZiBfbG9hZF93YXYocGF0aDogUGF0aCkgLT4gdHVwbGVbbnAubmRhcnJheSwgaW50XToKICAgIGF1ZGlvLCBzciA9IHNmLnJlYWQoc3RyKHBhdGgpLCBkdHlwZT0iZmxvYXQzMiIsIGFsd2F5c18yZD1UcnVlKQogICAgaWYgYXVkaW8uc2hhcGVbMV0gPiAxOgogICAgICAgIGF1ZGlvID0gYXVkaW8ubWVhbihheGlzPTEsIGtlZXBkaW1zPVRydWUpCiAgICByZXR1cm4gYXVkaW9bOiwgMF0sIHNyCgoKZGVmIGRpc2NvdmVyX2xpYnJpbWl4X3NhbXBsZXMoCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBzdWJzZXQ6IHN0ciA9ICJ0ZXN0IiwKICAgIG1heF9zYW1wbGVzOiBpbnQgfCBOb25lID0gTm9uZSwKKSAtPiBsaXN0W01peHR1cmVTYW1wbGVdOgogICAgIiIiCiAgICBEaXNjb3ZlciBMaWJyaU5NaXggc2FtcGxlcyAoTj0yLi41KSBmcm9tIGEgc3RhbmRhcmQgTGlicmlNaXggZGlyZWN0b3J5IGxheW91dC4KCiAgICBUaGUgbnVtYmVyIG9mIHNwZWFrZXJzIGlzIGRldGVjdGVkIGF1dG9tYXRpY2FsbHkgYnkgcHJvYmluZyB3aGljaCBzTi8KICAgIGRpcmVjdG9yaWVzIGV4aXN0IHVuZGVyIHRoZSBzdWJzZXQgZm9sZGVyLCBzbyB0aGUgc2FtZSBmdW5jdGlvbiB3b3JrcyBmb3IKICAgIExpYnJpMk1peCwgTGlicmkzTWl4LCBMaWJyaTRNaXgsIGFuZCBMaWJyaTVNaXggd2l0aG91dCBhbnkgZXh0cmEgYXJndW1lbnRzLgoKICAgIEV4cGVjdGVkIGxheW91dCAoMTYga0h6LCBtYXggbW9kZSk6CiAgICAgICAge2RhdGFfcm9vdH0vd2F2MTZrL21heC97c3Vic2V0fS9taXhfYm90aC8gICAjIGFsd2F5cyBwcmVzZW50CiAgICAgICAge2RhdGFfcm9vdH0vd2F2MTZrL21heC97c3Vic2V0fS9zMS8gICAgICAgICAjIE4gPj0gMQogICAgICAgIHtkYXRhX3Jvb3R9L3dhdjE2ay9tYXgve3N1YnNldH0vczIvICAgICAgICAgIyBOID49IDIKICAgICAgICB7ZGF0YV9yb290fS93YXYxNmsvbWF4L3tzdWJzZXR9L3MzLyAgICAgICAgICMgTiA+PSAzCiAgICAgICAge2RhdGFfcm9vdH0vd2F2MTZrL21heC97c3Vic2V0fS9zNC8gICAgICAgICAjIE4gPj0gNAogICAgICAgIHtkYXRhX3Jvb3R9L3dhdjE2ay9tYXgve3N1YnNldH0vczUvICAgICAgICAgIyBOID09IDUKCiAgICBBcmdzOgogICAgICAgIGRhdGFfcm9vdDogUm9vdCBvZiB0aGUgTGlicmlNaXggZGF0YXNldC4KICAgICAgICBzdWJzZXQ6IFNwbGl0IG5hbWUgKCd0cmFpbicsICdkZXYnLCAndGVzdCcpLgogICAgICAgIG1heF9zYW1wbGVzOiBDYXAgdGhlIG51bWJlciBvZiByZXR1cm5lZCBzYW1wbGVzLgoKICAgIFJldHVybnM6CiAgICAgICAgTGlzdCBvZiBNaXh0dXJlU2FtcGxlIG9iamVjdHMuCiAgICAiIiIKICAgIHJvb3QgPSBQYXRoKGRhdGFfcm9vdCkKICAgIHN1YnNldF9kaXIgPSByb290IC8gIndhdjE2ayIgLyAibWF4IiAvIHN1YnNldAogICAgbWl4X2RpciA9IHN1YnNldF9kaXIgLyAibWl4X2JvdGgiCiAgICBpZiBub3QgbWl4X2Rpci5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgZiJMaWJyaU1peCBtaXggZGlyZWN0b3J5IG5vdCBmb3VuZDoge21peF9kaXJ9XG4iCiAgICAgICAgICAgICJEb3dubG9hZCBMaWJyaU5NaXggYW5kIHNldCBkYXRhX3Jvb3QgaW4gY29uZmlncy9iYXNlbGluZS55YW1sLiIKICAgICAgICApCgogICAgbWl4X2ZpbGVzID0gc29ydGVkKG1peF9kaXIuZ2xvYigiKi53YXYiKSkKICAgIGlmIG1heF9zYW1wbGVzIGlzIG5vdCBOb25lOgogICAgICAgIG1peF9maWxlcyA9IG1peF9maWxlc1s6bWF4X3NhbXBsZXNdCgogICAgIyBBdXRvLWRldGVjdCBzcGVha2VyIGNvdW50IGZyb20gd2hpY2ggc04vIGRpcnMgZXhpc3QgKE49MS4uNSkuCiAgICAjIE9ubHkgcmFpc2UgaWYgdGhlcmUgYXJlIG1peCBmaWxlcyBidXQgbm8gc3RlbSBkaXJzIChjb3JydXB0ZWQgZGF0YXNldCkuCiAgICBtYXhfbiA9IHN1bSgxIGZvciBpIGluIHJhbmdlKDEsIDYpIGlmIChzdWJzZXRfZGlyIC8gZiJze2l9IikuaXNfZGlyKCkpCiAgICBpZiBtaXhfZmlsZXMgYW5kIG1heF9uIDwgMToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgZiJObyBzcGVha2VyIHN0ZW0gZGlyZWN0b3JpZXMgKHMxLy4uczUvKSBmb3VuZCB1bmRlciB7c3Vic2V0X2Rpcn0iCiAgICAgICAgKQoKICAgIHNhbXBsZXM6IGxpc3RbTWl4dHVyZVNhbXBsZV0gPSBbXQogICAgZm9yIG1peF9wYXRoIGluIG1peF9maWxlczoKICAgICAgICB1aWQgPSBtaXhfcGF0aC5zdGVtCiAgICAgICAgcmVmczogbGlzdFtucC5uZGFycmF5XSA9IFtdCiAgICAgICAgc3I6IGludCB8IE5vbmUgPSBOb25lCiAgICAgICAgZm9yIHNwa19pZHggaW4gcmFuZ2UoMSwgbWF4X24gKyAxKToKICAgICAgICAgICAgcmVmX3BhdGggPSBzdWJzZXRfZGlyIC8gZiJze3Nwa19pZHh9IiAvIGYie3VpZH0ud2F2IgogICAgICAgICAgICBpZiBub3QgcmVmX3BhdGguZXhpc3RzKCk6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICByZWYsIHJlZl9zciA9IF9sb2FkX3dhdihyZWZfcGF0aCkKICAgICAgICAgICAgaWYgc3IgaXMgTm9uZToKICAgICAgICAgICAgICAgIHNyID0gcmVmX3NyCiAgICAgICAgICAgIHJlZnMuYXBwZW5kKHJlZikKCiAgICAgICAgaWYgbm90IHJlZnMgb3Igc3IgaXMgTm9uZToKICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgbWl4dHVyZSwgbWl4X3NyID0gX2xvYWRfd2F2KG1peF9wYXRoKQogICAgICAgIGlmIG1peF9zciAhPSBzcjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlNhbXBsZSByYXRlIG1pc21hdGNoIGZvciB7dWlkfTogbWl4PXttaXhfc3J9LCByZWY9e3NyfSIpCgogICAgICAgIG1pbl9sZW4gPSBtaW4obGVuKG1peHR1cmUpLCAqKGxlbihyKSBmb3IgciBpbiByZWZzKSkKICAgICAgICBtaXh0dXJlID0gbWl4dHVyZVs6bWluX2xlbl0KICAgICAgICByZWZzX2FyciA9IG5wLnN0YWNrKFtyWzptaW5fbGVuXSBmb3IgciBpbiByZWZzXSwgYXhpcz0wKQoKICAgICAgICBzYW1wbGVzLmFwcGVuZCgKICAgICAgICAgICAgTWl4dHVyZVNhbXBsZSgKICAgICAgICAgICAgICAgIG1peHR1cmU9bWl4dHVyZS5hc3R5cGUobnAuZmxvYXQzMiksCiAgICAgICAgICAgICAgICByZWZlcmVuY2VzPXJlZnNfYXJyLmFzdHlwZShucC5mbG9hdDMyKSwKICAgICAgICAgICAgICAgIHNhbXBsZV9yYXRlPXNyLAogICAgICAgICAgICAgICAgdXR0ZXJhbmNlX2lkPXVpZCwKICAgICAgICAgICAgKQogICAgICAgICkKCiAgICByZXR1cm4gc2FtcGxlcwoKCmRlZiBpdGVyX21peHR1cmVzKAogICAgZGF0YV9yb290OiBzdHIgfCBQYXRoLAogICAgc3Vic2V0OiBzdHIgPSAidGVzdCIsCiAgICBtYXhfc2FtcGxlczogaW50IHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtNaXh0dXJlU2FtcGxlXToKICAgICIiIkFsaWFzIGZvciBkaXNjb3Zlcl9saWJyaW1peF9zYW1wbGVzIChiYXNlbGluZSBydW5uZXIgZW50cnkgcG9pbnQpLiIiIgogICAgcmV0dXJuIGRpc2NvdmVyX2xpYnJpbWl4X3NhbXBsZXMoZGF0YV9yb290LCBzdWJzZXQ9c3Vic2V0LCBtYXhfc2FtcGxlcz1tYXhfc2FtcGxlcykK'))
print('Project files extracted')


In [ ]:
import sys, subprocess
import shutil
shutil.copytree(SRCORRNET, '/tmp/sr_corrnet_src', dirs_exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','-e','/tmp/sr_corrnet_src','-q'],check=True)
subprocess.run([sys.executable,'-m','pip','install','soundfile','librosa','scipy','tqdm','-q'],check=True)
sys.path.insert(0, PROJ)
# Loguru stub
import os
os.makedirs('/tmp/loguru_stub/loguru', exist_ok=True)
open('/tmp/loguru_stub/loguru/__init__.py','w').write('from logging import getLogger as logger\nlogger = getLogger = __import__("logging").getLogger\n')
# Rotary stub
os.makedirs('/tmp/rotary_stub/rotary_embedding_torch', exist_ok=True)
open('/tmp/rotary_stub/rotary_embedding_torch/__init__.py','w').write('class RotaryEmbedding:\n    def __init__(self,*a,**k): pass\n    def __call__(self,*a,**k): return a[0] if a else None\n')
sys.path.insert(0, '/tmp/sr_corrnet_src')
sys.path.insert(0, '/tmp/loguru_stub')
sys.path.insert(0, '/tmp/rotary_stub')
print('installed')


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')


In [ ]:
# ── Stage 2 Eval ──────────────────────────────────────────────────────
import os, json as _json, numpy as np, torch, soundfile as sf
from pathlib import Path
from itertools import permutations
from train.stage1_single import _load_model, _get_inner_module
from models.lora import LoRALibrary
from data.calmsep_mixer import CalmSepMixer
from data.degradations import apply_reverb, apply_noise, apply_codec
from data.rir_bank import RirBank

SEED = 42

# ── Load model ──
ss_model = _load_model('shinuh/sr-corrnet-ss-1ch-wsj-var-2-5spk', torch.device('cpu'))
inner = _get_inner_module(ss_model)
lib = LoRALibrary(inner, adapter_names=['universal'])
lib.freeze_base()
inner.to(DEVICE)
engine = getattr(ss_model, 'engine', None)
if engine is not None:
    for _a in ('stft', 'istft'):
        _m = getattr(engine, _a, None)
        if _m is not None and hasattr(_m, 'to'): _m.to(DEVICE)
print('Model loaded')

# ── Load universal adapter ──
_ckpt_candidates = [
    STAGE2_CKPT,
    '/kaggle/working/checkpoints/stage2_universal/best_universal.pt',
]
_ckpt_path = next((c for c in _ckpt_candidates if os.path.exists(c)), None)
ADAPTER_LOADED = False
if _ckpt_path:
    ckpt = torch.load(_ckpt_path, map_location='cpu')
    state = ckpt.get('state_dict', ckpt)
    missing, _ = inner.load_state_dict(state, strict=False)
    print(f'Loaded adapter from {_ckpt_path}: {len(state)} tensors, {len(missing)} missing')
    ADAPTER_LOADED = True
else:
    print('WARNING: Stage 2 checkpoint not found — baseline-only eval')

# ── Build test mixer ──
_libri = Path(AUDIO)
_src = sorted(_libri.rglob('*.flac')) + sorted(_libri.rglob('*.wav'))
_held = set()
_mf = _libri / 'manifest_8k.json'
if _mf.exists():
    _d = _json.loads(_mf.read_text())
    _items = _d if isinstance(_d, list) else list(_d.get('splits', {}).values())
    for _si in _items:
        if isinstance(_si, dict) and ('test' in _si.get('split','') or 'dev' in _si.get('split','')):
            _held.update(_si.get('speaker_ids', _si.get('speakers', [])))
print(f'Test speakers held out: {len(_held)}')
_rng = np.random.default_rng(SEED)
mixer = CalmSepMixer(_src, held_out_speaker_ids=_held, rng=_rng)

# ── RIR bank ──
_rir_dir = _libri / 'rirs'
rir_bank = RirBank(_rir_dir) if (_rir_dir / 'bank.json').exists() else None
print(f'RIR bank: {"loaded" if rir_bank else "not found"}')

# ── Noise files ──
_nd = _libri / 'noise'
noise_files = []
if _nd.exists():
    noise_files = sorted((_nd/'wham').glob('*_8k.wav')) + sorted((_nd/'dns4').glob('*_8k.wav'))
    if not noise_files: noise_files = sorted(_nd.rglob('*_8k.wav'))
print(f'Noise files: {len(noise_files)}')


In [ ]:
# ── SI-SNR helpers ──
def _si_snr(est, ref):
    t = min(est.shape[-1], ref.shape[-1])
    e = est[..., :t]; e = e - e.mean(-1, keepdim=True)
    r = ref[..., :t]; r = r - r.mean(-1, keepdim=True)
    dot = (e * r).sum(-1, keepdim=True)
    s = dot / (r.pow(2).sum(-1, keepdim=True) + 1e-10) * r
    return (10 * torch.log10(s.pow(2).sum(-1) / ((e - s).pow(2).sum(-1) + 1e-10) + 1e-10)).item()

def _pit(est, ref):
    K, N = est.shape[0], ref.shape[0]
    best = -1e9
    for perm in permutations(range(N)):
        v = sum(_si_snr(est[k], ref[perm[k]]) for k in range(min(K, N))) / min(K, N)
        if v > best: best = v
    return best

def _mix_snr(mix, ref):
    return sum(_si_snr(mix, ref[k]) for k in range(ref.shape[0])) / ref.shape[0]

# ── Inference helper ──
def _infer(wav_t, use_adapter):
    if use_adapter and ADAPTER_LOADED:
        lib.set_adapter('universal', co_activate=False)
    else:
        lib.set_gates({'universal': 0.0})
    lib.inject_gates()
    with torch.inference_mode():
        out = ss_model.process_waveform(wav_t, n_spks=None)
    wavs = out.get('waveforms', [])
    if not wavs: return None
    return torch.stack([
        (w if isinstance(w, torch.Tensor) else torch.from_numpy(w)).squeeze().to(DEVICE)
        for w in wavs
    ])

# ── Eval loop ──
eval_rng = np.random.default_rng(SEED + 1)
results = {c: {'baseline': [], 'universal': []} for c in ['clean','reverb','noise','codec']}

for cond in ['clean', 'reverb', 'noise', 'codec']:
    print(f'\n  [{cond}]', flush=True)
    for i in range(N_EVAL_PER_COND):
        m = mixer.mix(split='test')
        try:
            if cond == 'reverb' and rir_bank: m = apply_reverb(m, rir_bank, eval_rng)
            elif cond == 'noise' and noise_files:
                nf = noise_files[int(eval_rng.integers(len(noise_files)))]
                nw, _ = sf.read(str(nf), dtype='float32')
                m = apply_noise(m, nw, eval_rng)
            elif cond == 'codec': m = apply_codec(m, 'opus', 12_000)
        except Exception: pass

        mix_t = torch.from_numpy(m.mixture).float().unsqueeze(0).to(DEVICE)
        ref_t = torch.from_numpy(m.references).float().to(DEVICE)
        mix_snr = _mix_snr(mix_t.squeeze(0), ref_t)

        for mode, use_adp in [('baseline', False), ('universal', True)]:
            est = _infer(mix_t, use_adp)
            if est is not None:
                results[cond][mode].append(_pit(est, ref_t) - mix_snr)
        if (i+1) % 10 == 0: print(f'    {i+1}/{N_EVAL_PER_COND}', flush=True)

print('\nDone.')


In [ ]:
# ── Results table ──
print('\n' + '='*72)
print('STAGE 2 UNIVERSAL ADAPTER EVALUATION')
print('='*72)
print(f'{"Condition":<10} {"Baseline SI-SDRi":>18} {"Universal SI-SDRi":>19} {"Delta":>9}')
print('-'*72)
for cond in ['clean','reverb','noise','codec']:
    b = results[cond]['baseline']
    u = results[cond]['universal']
    b_m = np.mean(b) if b else float('nan')
    u_m = np.mean(u) if u else float('nan')
    delta = u_m - b_m if (b and u) else float('nan')
    print(f'{cond:<10} {b_m:>14.2f} dB    {u_m:>14.2f} dB   {delta:>+8.2f} dB')
print('='*72)

# BLUEPRINT decision
deltas = {c: (np.mean(results[c]['universal']) - np.mean(results[c]['baseline']))
          if results[c]['universal'] else float('nan')
          for c in ['clean','reverb','noise','codec']}

print('\nBLUEPRINT §8.3 DECISION GATE:')
print(f'  Clean delta  : {deltas["clean"]:+.2f} dB  (must be >= -0.1 dB)')
print(f'  Reverb delta : {deltas["reverb"]:+.2f} dB')
print(f'  Noise delta  : {deltas["noise"]:+.2f} dB')
print(f'  Codec delta  : {deltas["codec"]:+.2f} dB')
print()
worst = min(v for v in deltas.values() if not np.isnan(v))
if worst >= -0.5:
    print('  UNIVERSAL ADAPTER IS COMPETITIVE (worst delta >= -0.5 dB).')
    print('  Compare to Stage 1 per-condition adapters (Stage 1 reverb+noise done).')
    print('  If Stage 1 adapters also within 0.5 dB -> adopt universal, skip Stage 3.')
else:
    print('  VERDICT: Universal adapter has gap > 0.5 dB on at least one condition.')
    print('  Proceed to Stage 3 (gate/routing network) as planned.')

# Save JSON
import json
out = {c: {'baseline': results[c]['baseline'], 'universal': results[c]['universal']} for c in results}
json.dump(out, open('/kaggle/working/stage2_eval_results.json','w'), indent=2)
print('\nResults saved to /kaggle/working/stage2_eval_results.json')
